# Hafta 14 · 1. bölüm · Büyük Dil Modelleri I: Nasıl Eğitilir, Nasıl Çalışır?

**Yapay Zekâya Giriş — Laboratuvar Defteri**  
Dr. Seda Bayat Toksöz · Iğdır Üniversitesi · 2026–2027 Güz

Bu defterde dil modellerinin parçalarını kendi elinizle kuracaksınız: (P1) sıfırdan bir BPE tokenleştirici eğitip Türkçe ile İngilizcenin token maliyetini karşılaştıracak, (P6) eğitim belleği ve maliyet hesaplayıcısı yazacak, (P2) ders notlarımızın metniyle küçük bir karakter düzeyinde GPT'yi ön eğitecek, (P3) aynı modelle kod çözme stratejilerini deneyecek, (P4) KV önbelleğinin hızını ölçecek, (P5) modeli 8 ve 4 bite nicemleyecek, (P2b) “Soru:/Cevap:” örnekleriyle küçük bir talimat ince ayarı (SFT) yapacak ve akıl yürütme modellerindeki çoğunluk oyu ile GRPO avantajını hesaplayacaksınız. Eğitim hücreleri CPU'da birkaç dakika sürer; Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU seçerseniz hızlanır.

Hücreleri yukarıdan aşağıya sırayla çalıştırın (**Shift + Enter**). Parametreleri değiştirip sonucun nasıl değiştiğini gözlemleyin.

In [1]:
# Grafik ayarları (ders notlarıyla aynı renk paleti)
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=["#0072B2", "#D55E00", "#009E73", "#E69F00", "#56B4E9", "#CC79A7"]),
})

## 0. Proje verisi: ders notlarımızın metni
Ders notlarımızdaki düzyazı paragraflardan alınmış ≈ 100 bin karakterlik Türkçe metin. Tüm projeler bu metni kullanır; hücreyi çalıştırmanız yeterli.

In [2]:
# Ders notlarımızdan ≈100 bin karakterlik Türkçe metin (proje verisi)
metin = """Yapay zekâ, son on yılda bilgisayar biliminin en hızlı gelişen alanı oldu. Telefonunuzun kilidini açan yüz tanıma sistemi, e-posta kutunuzdaki istenmeyen posta filtresi, hastanelerde tümörleri işaretleyen görüntü analiz yazılımları ve sohbet ettiğiniz büyük dil modelleri aynı temel fikre dayanır: verilerden öğrenen modeller. Bu ders notları, Yazılım Mühendisliği 4. sınıf öğrencilerinin bu fikri hem kavramsal hem de hesaplama düzeyinde kavraması için hazırlandı. Notlar 14 haftalık ders planına göre düzenlenmiştir. İlk yedi hafta makine öğrenmesinin ve yapay sinir ağlarının temellerini kapsar: veri ve öğrenme türleri, regresyon, sınıflandırma, model değerlendirme, sinir ağları ve geri yayılım. İkinci yarıda bu temellerin üzerine görüntü işleme, evrişimli sinir ağları (CNN), görüntü segmentasyonu, doğal dil işleme, Transformer mimarisi ve büyük dil modelleri (LLM) inşa edilir. Her hafta bir öncekinin üzerine kurulduğu için konuları sırasıyla ça-
Bu dersteki amacımız formül ezberlemek değil, bir modelin neden ve nasıl çalıştığını anlamaktır. Bu yüzden her haftada adım adım çözülmüş sayısal örnekler bulacaksınız. Bu örnekleri yalnızca okumayın; kâğıt, kalem ve hesap makinesiyle kendiniz de çözün. Sınavlardaki hesaplama soruları bu örneklerin benzerleridir. Haftaların içindeki kısa Python kodları ise laboratuvar uygulamalarına hazırlık niteliğindedir; kodları kendi bilgisayarınızda ya da Google Colab üzerinde çalıştırarak sonuçları notlardaki elle yapılmış hesaplarla karşılaştırmanız, konuyu pekiştirmenin en iyi yoludur.
Vize ve final sınavları bu notlardaki konular, tanımlar, çözümlü örnekler ve her haftanın sonundaki değerlendirme soruları esas alınarak hazırlanır. Önerilen vize kapsamı 1-7. haftalardır; final sınavı tüm haftaları kapsar.
Her hafta aynı yapıyı izler. Haftanın başında o haftanın kazanımları ve anahtar kavramları listelenir; ardından konu anlatımı, şekiller ve çözümlü örnekler gelir. Hafta, kısa bir özet ve değerlendirme soruları ile biter. Sorular üç gruptur: (A) çoktan seçmeli sorular, (B) kısa cevaplı ve hesaplama soruları, (C) birden çok kavramı birleştiren ileri düzey sorular. Soruların cevapları Ek D'deki cevap anahtarında yer alır; önce kendiniz çözmeye çalışın, sonra kontrol edin. Metin içinde farklı amaçlara hizmet eden renkli kutular kullanılmıştır:
Konunun matematiksel arka planını, türetmeleri ve ileri düzey hesaplamaları içeren kutulardır. Temel anlatımı kesintiye uğratmadan bir adım daha derine inmek isteyenler içindir; sınavlardaki ileri düzey sorular bu kutulardaki içerikten yararlanır.
- Yuvarlama: Ara adımlarda genellikle 3-4 anlamlı basamak tutulmuştur. Sınavda kendi yuvarlamanızdan kaynaklanan küçük farklar sorun değildir. - Terimler: Türkçe terimlerin İngilizce karşılıkları ilk geçtikleri yerde parantez içinde verilmiştir. Tüm terimler Ek C'deki sözlükte alfabetik olarak toplanmıştır. - Görseller: Fotoğraf örnekleri, açık kaynaklı scikit-image kütüphanesinin örnek görüntülerinden alınmıştır. Diğer tüm şekil ve grafikler bu notlar için hazırlanmıştır.
Bu ders; yapay zekânın temel kavramlarını, makine öğrenmesi ve derin öğrenme yöntemlerini, görüntü ve metin verileri üzerinde çalışan modern mimarileri (CNN, Transformer, büyük dil modelleri) sezgisel ve hesaplamalı bir bakışla tanıtmayı amaçlar. Ders sonunda öğrencilerin bir yapay
zekâ problemini doğru öğrenme türüyle eşleştirebilmesi, temel modelleri elle ve kodla uygulayabilmesi ve modellerin başarımını doğru metriklerle yorumlayabilmesi hedeflenir.
- Her dersten önce ilgili haftanın kazanımlarını ve özetini okuyun; dersten sonra anlatımı ve çözümlü örnekleri ayrıntılı çalışın. - Çözümlü örnekleri önce kendiniz çözün, sonra çözümle karşılaştırın. - Değerlendirme sorularını cevap anahtarına bakmadan çözün; yanlış yaptığınız soruların konu-
- Python kodlarını çalıştırın, parametreleri değiştirerek sonucun nasıl değiştiğini gözlemleyin. - Ek B'deki formül özetini sınavdan önce tekrar listesi olarak kullanın.
Bu haftanın sonunda öğrenciler; - yapay zekânın farklı tanımlarını açıklayabilecek ve dar, genel ve süper yapay zekâ ayrımını
- geleneksel programlama ile makine öğrenmesi yaklaşımını karşılaştırabilecek, - yapay zekânın tarihsel gelişimindeki kilometre taşlarını sıralayabilecek, - akıllı ajan kavramını PEAS çerçevesiyle tanımlayabilecek, - A gibi arama algoritmalarının çalışma mantığını açıklayıp küçük bir haritada uygulaya-
"Zekâ" kelimesini tanımlamak bile kolay değildir: Bir satranç ustasının hamle seçmesi, bir çocuğun yürümeyi öğrenmesi, bir doktorun röntgen filminde kırığı fark etmesi ya da bir arkadaşınızın şakanızı anlaması farklı zekâ biçimleridir. Yapay zekâ alanı bu yeteneklerin bir bölümünü bilgisayarlarla gerçekleştirmeyi amaçlar. Terimi ilk kez kullanan John McCarthy yapay zekâyı "akıllı makineler, özellikle de akıllı bilgisayar programları yapma bilimi ve mühendisliği" olarak tanımlamıştır.
Yapay zekâ (artificial intelligence, AI), normalde insan zekâsı gerektiren görevleri -algılama, öğrenme, akıl yürütme, problem çözme, karar verme ve dil anlama gibi- yerine getirebilen bilgisayar sistemleri geliştirmeyi amaçlayan bilim ve mühendislik dalıdır.
Yapay zekâ ders kitaplarının en bilineni olan Russell ve Norvig'in kitabı, alandaki tanımları iki soruya göre dört gruba ayırır: Sistem düşünmeye mi yoksa davranmaya mı odaklanıyor? Başarı ölçütü insana benzemek mi yoksa rasyonel (en doğru) olmak mı? Tablo 1.1 bu dört yaklaşımı özetler.
Alan Turing, 1950 yılında yayımladığı "Computing Machinery and Intelligence" makalesinde "Makineler düşünebilir mi?" sorusunun belirsiz olduğunu savunarak yerine ölçülebilir bir deney önermiştir: taklit oyunu (imitation game). Bugün Turing testi (Turing test) olarak bilinen bu deneyde bir sorgulayıcı, göremediği iki tarafla yalnızca yazılı mesajlarla konuşur (Şekil 1.1). Taraflardan biri insan, diğeri makinedir. Sorgulayıcı hangisinin makine olduğunu güvenilir biçimde ayırt edemiyorsa makine testi geçmiş sayılır.
Filozof John Searle (1980), Turing testine karşı ünlü "Çin Odası" düşünce deneyini önermiştir: Çince bilmeyen bir kişi, bir kural kitabını izleyerek kendisine verilen Çince sorulara kusursuz Çince cevaplar üretebilir; ancak yine de söylediklerinin hiçbirini anlamaz. Searle'e göre davranışı taklit etmek, anlamak demek değildir. Bu tartışma, büyük dil modelleri çağında yeniden
Yapay zekâ sistemleri yeteneklerinin genişliğine göre üç düzeyde incelenir (Tablo 1.2). Günümüzde kullanılan sistemlerin büyük çoğunluğu belirli bir görev için tasarlanmış dar yapay zekâ (narrow AI) örnekleridir. Büyük dil modelleri çok sayıda farklı görevi yerine getirebildikleri için "genel amaçlı" olarak nitelendirilse de insan düzeyinde genel yapay zekâya (artificial general intelligence, AGI) ulaşılıp ulaşılmadığı hâlâ tartışma konusudur.
Bu üç terim günlük dilde çoğu zaman birbirinin yerine kullanılır; oysa aralarında iç içe bir ilişki vardır (Şekil 1.2). Yapay zekâ en geniş kavramdır ve öğrenmeyen sistemleri de (örneğin el ile yazılmış kurallarla çalışan uzman sistemler veya arama algoritmaları) kapsar. Makine öğrenmesi yapay zekânın, derin öğrenme de makine öğrenmesinin bir alt alanıdır.
Makine öğrenmesi (machine learning, ML), bilgisayarların açıkça programlanmadan, verilerden ve deneyimden öğrenerek bir görevdeki başarımlarını artırmasını sağlayan yöntemleri incele-
Tom Mitchell (1997) daha biçimsel ve sınavlarda sıkça sorulan bir tanım vermiştir: Bir bilgisayar programı, T görevlerindeki P ölçütüyle ölçülen başarımı E deneyimiyle artıyorsa, bu program E deneyiminden öğreniyor denir. Burada T görev (task), P performans ölçütü (performance), E ise deneyimdir
- E (deneyim): Ziraat uzmanlarınca etiketlenmiş binlerce yaprak fotoğrafı. - P (performans): Daha önce görülmemiş test fotoğraflarında doğru sınıflandırılanların
Geleneksel programlamada yazılım mühendisi problemi çözen kuralları kendisi yazar; bilgisayar bu kuralları verilere uygulayarak cevap üretir. Makine öğrenmesinde ise bilgisayara veriler ve bu verilere ait doğru cevaplar (etiketler) verilir; bilgisayar bu ikisi arasındaki ilişkiyi, yani kuralları, kendisi çıkarır (Şekil 1.3). Öğrenilen bu kurallar kümesine model (model) denir ve model daha sonra yeni veriler için tahmin yapmakta kullanılır.
Bu farkı bir istenmeyen e-posta (spam) filtresi üzerinden düşünelim. Kural tabanlı bir filtre "konu satırında kazandınız geçiyorsa spam olarak işaretle" gibi yüzlerce kural içerir. Bu yaklaşımın üç temel sorunu vardır: (i) kurallar zamanla çoğalır ve bakımı zorlaşır, (ii) spam gönderenler kelimeleri değiştirerek ("kazndınız") kuralları kolayca atlatır, (iii) her dil ve her yeni saldırı türü için kural yazmak gerekir. Makine öğrenmesi tabanlı bir filtre ise kullanıcıların "spam" olarak işaretlediği binlerce e-postadan hangi kelime ve özelliklerin spam ile ilişkili olduğunu kendisi öğrenir ve yeni örnekler geldikçe yeniden eğitilerek güncellenir.
Derin öğrenme (deep learning, DL), çok sayıda katmandan oluşan yapay sinir ağlarıyla verinin basitten karmaşığa doğru hiyerarşik temsillerini otomatik olarak öğrenen makine öğrenmesi
Klasik makine öğrenmesi ile derin öğrenme arasındaki en önemli fark, öznitelik (feature) adı verilen ve modele girdi olarak verilen ölçülebilir özelliklerin nasıl elde edildiğidir. Klasik yaklaşımda uzman bir insan, örneğin bir görüntüden kenar yoğunluğu, renk histogramı veya doku ölçüleri gibi öznitelikleri elle tasarlar (öznitelik mühendisliği (feature engineering)). Derin öğrenmede ise ağ, ham pikselleri girdi olarak alır ve hangi özniteliklerin önemli olduğunu eğitim sırasında kendisi öğrenir
Son yıllarda en çok konuşulan alt alan ise üretken yapay zekâdır (generative AI): Bu modeller yalnızca sınıflandırma veya tahmin yapmakla kalmaz, eğitim verisine benzeyen yeni metin, görüntü, ses ve kod üretir. ChatGPT, Gemini ve Claude gibi sohbet asistanlarının temelindeki büyük dil modellerini 14. haftada ayrıntılı olarak inceleyeceğiz.
Yapay zekânın tarihi düz bir çizgide ilerlememiş; büyük beklentilerle geçen "bahar" dönemleri ile hayal kırıklığı ve fon kesintileriyle geçen YZ kışları (AI winters) birbirini izlemiştir. Şekil 1.5 başlıca kilometre taşlarını özetlemektedir. Tarihçeden çıkarılacak en önemli ders şudur: Derin öğrenmenin 2012'den sonraki yükselişi tek bir buluşa değil, üç etkenin aynı anda olgunlaşmasına dayanır.
2. Hesaplama gücü: Başlangıçta bilgisayar oyunları için geliştirilen grafik işlem birimleri (GPU), sinir ağlarının ihtiyaç duyduğu paralel matris çarpımlarını çok hızlı yapabiliyordu. 3. Algoritmik gelişmeler: ReLU aktivasyonu, dropout, daha iyi ağırlık başlatma ve optimizasyon yöntemleri derin ağların eğitimini kararlı hâle getirdi.
2024 Nobel Fizik Ödülü, yapay sinir ağlarıyla makine öğrenmesini mümkün kılan temel keşifleri nedeniyle John Hopfield ve Geoffrey Hinton'a verildi. Aynı yıl Nobel Kimya Ödülü'nün yarısı protein yapısı tahmini yapan AlphaFold modeli için Demis Hassabis ve John Jumper'a, diğer yarısı ise hesaplamalı protein tasarımı için David Baker'a verildi. Yapay zekâ böylece temel bilimlerin en prestijli ödüllerine konu oldu.
Modern yapay zekâda sistemler genellikle ajan (agent) kavramıyla modellenir. Ajan, çevresini algılayıcılar aracılığıyla algılayan ve çevresi üzerinde eyleyiciler aracılığıyla eylemde bulunan her şeydir (Şekil 1.6). Bir robot süpürge, bir satranç programı ya da bir sohbet asistanı birer ajandır. Bir ajan tasarlarken önce görev ortamını tanımlamak gerekir. Bunun için kullanılan çerçeveye, İngilizce bileşen adlarının baş harflerinden oluşan PEAS (Performance, Environment, Actuators, Sensors) denir. Bu çerçeve dört soruyu yanıtlar: Başarı neyle ölçülecek (performans ölçütü)? Ajan nerede çalışacak (ortam)? Hangi araçlarla eylemde bulunacak (eyleyiciler)? Çevreyi nasıl algılayacak
Ajanın çalıştığı ortamın özellikleri tasarım zorluğunu belirler. Ortam tam gözlemlenebilir (satranç tahtasının tamamı görünür) ya da kısmi gözlemlenebilir (taksi, köşenin arkasını göremez); deterministik (aynı hamle hep aynı sonucu verir) ya da stokastik (yayanın ne yapacağı kesin değildir); statik ya da dinamik olabilir. Otonom sürüş; kısmi gözlemlenebilir, stokastik ve dinamik bir ortam olduğu için en zor yapay zekâ problemlerinden biridir.
Makine öğrenmesi yaygınlaşmadan önce yapay zekânın temel aracı arama (search) idi ve bugün de rota planlama, oyun oynama ve otomatik planlama gibi birçok problem arama ile çözülür. Bir arama probleminde bir başlangıç durumu, her durumda uygulanabilecek eylemler, eylemlerin maliyetleri ve bir hedef vardır. Amaç, başlangıçtan hedefe giden en düşük maliyetli eylem dizisini bulmaktır. Haritada iki şehir arasındaki en kısa yol, 8-bulmacasının çözümü veya bir robotun engellerin arasından geçişi bu biçimde modellenir. Arama algoritmaları iki gruba ayrılır. Bilgisiz (uninformed) algoritmalar hedefe ne kadar yakın olduklarını bilmeden ilerler: Genişlik öncelikli arama (BFS) durumları başlangıçtan uzaklıklarına göre katman katman gezer; derinlik öncelikli arama (DFS) bir dalı sonuna kadar izler; tek biçimli maliyet araması (UCS) her adımda o ana kadarki toplam maliyeti g(n) en küçük olan durumu genişletir. Bilgili (informed) algoritmalar ise bir sezgisel fonksiyon (heuristic function) h(n) kullanır: h(n), n durumundan hedefe kalan maliyetin bir tahminidir (ör. haritada kuş uçuşu uzaklık). En ünlü bilgili arama algoritması olan A araması (A search), her adımda
değeri en küçük olan durumu genişletir. Sezgisel fonksiyon gerçek kalan maliyeti hiçbir zaman aşmıyorsa (kabul edilebilir sezgisel), A en düşük maliyetli çözümü bulmayı garanti eder.
Aşağıdaki haritada şehirler arası yol maliyetleri ve her şehrin hedef G'ye kuş uçuşu uzaklık tahmini h verilmiştir. A algoritmasıyla S'den G'ye en ucuz yolu bulunuz.
dönüşür ve aynı sonucu genellikle daha fazla düğüm genişleterek bulur (bu küçük haritada her iki yöntem de beş düğümün tamamını genişletir; fark büyük haritalarda ortaya çıkar).
Arama, problemin kurallarının açıkça bilindiği durumlarda (harita, satranç, bulmaca) çok güçlüdür; öğrenme ise kuralların veriden çıkarılması gerektiğinde devreye girer. Modern sistemler ikisini birleştirir: AlphaGo, hamle olasılıklarını derin bir ağla tahmin eder, en iyi hamleyi ise bu tahminlerle yönlendirilen bir ağaç araması bulur.
Yapay zekâ bugün hemen her sektörde kullanılmaktadır. Şekil 1.7 bu dönem boyunca inceleyeceğimiz yöntemlerin gerçek hayattaki karşılıklarından bazılarını göstermektedir.
Yapay zekâ sistemleri insanların hayatını doğrudan etkileyen kararlarda (işe alım, kredi onayı, tıbbi teşhis) kullanıldıkça etik sorular da önem kazanmaktadır. Bir yazılım mühendisi olarak geliştireceğiniz sistemlerde aşağıdaki ilkeleri göz önünde bulundurmanız gerekir:
- Yanlılık ve adalet: Model, eğitim verisindeki önyargıları öğrenir ve büyütebilir. Örneğin geçmişte çoğunlukla erkek adayların işe alındığı bir şirketin verileriyle eğitilen bir özgeçmiş değerlendirme sistemi, kadın adayları sistematik olarak düşük puanlayabilir. Bu duruma algoritmik yanlılık (algorithmic bias) denir. - Gizlilik ve veri koruma: Kişisel verilerin toplanması, saklanması ve işlenmesi yasal düzenlemelere uygun olmalıdır. Türkiye'de bu konuda 6698 sayılı Kişisel Verilerin Korunması Kanunu
- Şeffaflık ve açıklanabilirlik: Özellikle derin öğrenme modelleri "kara kutu" gibi davranabilir. Bir kredi başvurusunu reddeden modelin bu kararı neden verdiğini açıklayabilmesi gerekir. Bu ihtiyaç açıklanabilir yapay zekâ (explainable AI, XAI) alanını doğurmuştur. - Sorumluluk: Otonom bir aracın karıştığı kazada sorumluluğun kime (üretici, yazılımcı, kullanıcı) ait olduğu hukuki bir sorudur. Avrupa Birliği'nin Yapay Zekâ Yasası (AI Act), sistemleri risk düzeyine göre sınıflandıran ilk kapsamlı düzenlemelerden biridir. - Kötüye kullanım: Gerçeğinden ayırt edilmesi zor sahte ses ve videolar (deepfake), dezenformasyon ve otomatik siber saldırılar ciddi riskler oluşturur. - Akademik dürüstlük: Ödev ve projelerde üretken yapay zekâ araçlarını kullanırken ders kurallarına uymalı, kullanımı açıkça belirtmeli ve üretilen içeriğin doğruluğunu mutlaka kontrol
"Çöp girer, çöp çıkar" (garbage in, garbage out) ilkesi yapay zekâda da geçerlidir. Hatalı, eksik veya yanlı verilerle eğitilen bir model, ne kadar karmaşık olursa olsun güvenilir sonuçlar
Bu derste göreceğimiz hemen her model; girdileri sayı listeleri, yani vektörler (vectors) olarak alır ve bunlar üzerinde matris işlemleri yapar. Bir görüntü piksel değerlerinden, bir cümle kelime temsillerinden, bir hasta kaydı ise ölçüm değerlerinden oluşan bir vektöre dönüştürülür. Bu nedenle ilk haftada iki temel işlemi hatırlayalım (ayrıntılı tekrar için Ek A'ya bakınız). İç çarpım (dot product): Aynı boyuttaki iki vektörün karşılıklı elemanları çarpılıp toplanır:
(a) şıkkındaki işlem, 6. haftada göreceğimiz yapay nöronun yaptığı hesabın ta kendisidir: girdileri ağırlıklarıyla çarp, topla ve bias ekle. (c) şıkkı ise aynı anda iki nöronun hesabını yapar.
Yapay zekâ, insan zekâsı gerektiren görevleri yerine getiren sistemler geliştirmeyi amaçlar; modern yaklaşım rasyonel ajan yaklaşımıdır. Turing testi, makinenin yazışmalarda insandan ayırt edilemeyecek davranış sergileyip ser-
Derin öğrenme devrimi; büyük veri, GPU'larla gelen hesaplama gücü ve algoritmik gelişmelerin birleşimiyle 2012'de başladı. Ajan, algılayıcılarıyla çevreyi algılar ve eyleyicileriyle çevreye etki eder; görev ortamı PEAS
Yanlılık, gizlilik, açıklanabilirlik ve sorumluluk, yapay zekâ sistemleri tasarlanırken gözetilmesi gereken temel etik ilkelerdir.
3. Makine öğrenmesi yaklaşımında sisteme verilenler ve sistemin ürettiği çıktı aşağıdakilerin hangisinde doğru eşleştirilmiştir?
4. Mitchell'in makine öğrenmesi tanımına göre, bir spam filtresi için "doğru sınıflandırılan epostaların yüzdesi" hangi bileşene karşılık gelir?
A Abartılı beklentilerin karşılanamaması B Dönemin bilgisayarlarının hesaplama gücünün yetersiz olması C Tek katmanlı perceptronun XOR problemini çözemediğinin gösterilmesi D Araştırma fonlarının kesilmesi E Transformer modellerinin eğitim maliyetinin çok yüksek olması
7. Aşağıdakilerden hangisi dar (zayıf) yapay zekâya örnektir? A Her alanda en yetenekli insanları aşan bir sistem B Yalnızca yaprak fotoğraflarından bitki hastalığı tespit eden bir model C İnsan gibi her türlü görevi öğrenip bilgisini başka alanlara aktarabilen bir sistem D Bilinç sahibi olduğu kanıtlanmış bir robot
12. Mitchell'in T-P-E tanımını, bir e-ticaret sitesinde kullanıcılara ürün öneren bir sistem için yazınız. Bu sistemin başarımını yalnızca "tıklama oranı" ile ölçmenin hangi etik ve teknik sorunlara yol açabileceğini tartışınız.
Bu haftanın sonunda öğrenciler; - bir veri setinde örnek, öznitelik ve etiket kavramlarını ayırt edebilecek, - denetimli, denetimsiz ve pekiştirmeli öğrenmeyi örneklerle açıklayabilecek; bir problemi doğru öğrenme türüyle eşleştirebilecek, - sınıflandırma ile regresyon problemlerini ayırt edebilecek, - makine öğrenmesi iş akışının adımlarını sıralayabilecek, - veriyi eğitim, doğrulama ve test kümelerine ayırmanın amacını ve veri sızıntısı riskini açık-
- aşırı ve eksik öğrenmeyi tanımlayabilecek, - min-maks normalizasyonu, z-skoru standardizasyonu ve one-hot kodlamayı elle uygula-
- Pearson korelasyon katsayısını hesaplayıp yorumlayabilecek; dengesiz veri setleri için yeniden örnekleme ve sınıf ağırlığı yaklaşımlarını açıklayabilecektir.
Makine öğrenmesi modelleri kurallarını veriden çıkardığı için verinin yapısını doğru anlamak her şeyin başlangıcıdır. Tablo biçimindeki bir veri setinde (dataset) her satır bir örnek (sample, instance) (gözlem), her sütun bir öznitelik ve tahmin etmek istediğimiz sütun ise etiket (label) ya da hedef
Belirleyici olan girdinin değil, çıktının (etiketin) türüdür. "Bir hastanın tümörü iyi huylu mu, kötü huylu mu?" sorusu sınıflandırma; "tümörün çapı kaç milimetre?" sorusu ise regresyondur. Çıktı sayı olsa bile sınıf belirtiyorsa (ör. rakam tanımada "7") problem sınıflandırmadır.
Denetimsiz öğrenmede (unsupervised learning) verilerin etiketi yoktur. Model, verinin içindeki yapıyı, grupları veya örüntüleri kendi başına keşfetmeye çalışır.
En yaygın denetimsiz görevler şunlardır: benzer örnekleri gruplayan kümeleme (clustering) (ör. müşteri segmentasyonu), çok boyutlu veriyi daha az boyutla temsil eden boyut indirgeme (dimensionality reduction) ve olağan dışı örnekleri bulan anomali tespiti (anomaly detection). Şekil 2.2 denetimli ve denetimsiz öğrenmeyi karşılaştırmaktadır.
Pekiştirmeli öğrenmede (reinforcement learning) bir ajan, çevresiyle deneme-yanılma yoluyla etkileşime girer. Her eyleminin ardından bir ödül (reward) (ya da ceza) alır ve uzun vadede toplam ödülü en üst düzeye çıkaran davranış stratejisini (policy) öğrenir.
Pekiştirmeli öğrenmede doğru cevap önceden verilmez; ajan yalnızca yaptığı şeyin ne kadar iyi olduğunu ödül sinyalinden anlar (Şekil 2.3). Oyun oynayan programlar (AlphaGo), robotların yürümeyi öğrenmesi ve büyük dil modellerinin insan tercihleriyle hizalanması (RLHF, 14. hafta) bu
Etiketleme çoğu zaman pahalıdır: Bir radyoloğun binlerce MR görüntüsünü tek tek işaretlemesi haftalar sürer. Yarı denetimli öğrenme (semi-supervised learning), az sayıda etiketli örneği çok sayıda etiketsiz örnekle birlikte kullanır. Öz-denetimli öğrenmede (self-supervised learning) ise etiket verinin kendisinden üretilir. Örneğin bir cümlenin sonraki kelimesini tahmin etmek için etikete gerek yoktur, çünkü sonraki kelime metnin içinde zaten vardır. Büyük dil modellerinin ön eğitimi tam
Gerçek bir makine öğrenmesi projesi yalnızca "model eğitmekten" ibaret değildir. Şekil 2.4'deki adımlar bir yazılım projesinin yaşam döngüsüne benzer ve çoğu zaman tekrarlıdır: Değerlendirme sonuçları beklentiyi karşılamazsa veri toplama veya öznitelik adımlarına geri dönülür.
2. Veri toplama: Yeterli miktarda, problemi temsil eden ve mümkünse etiketli veri toplanır. 3. Veri ön işleme: Eksik ve hatalı değerler düzeltilir, veri ölçeklenir ve kodlanır (Bölüm 2.6). 4. Öznitelik mühendisliği: Modelin işini kolaylaştıracak yeni öznitelikler türetilir (ör. doğum ta-
5. Model eğitimi: Uygun algoritma seçilir ve eğitim verisiyle eğitilir. 6. Değerlendirme: Model, eğitimde hiç görmediği test verisi üzerinde uygun metriklerle ölçülür
7. Dağıtım ve izleme: Model bir uygulamaya entegre edilir; zamanla verinin değişmesi (data drift) nedeniyle başarımı izlenir ve gerektiğinde yeniden eğitilir. Bu adımların yazılım mühendisliği disipliniyle yönetilmesine MLOps (machine learning operations) denir.
Sektörde yapılan anketlerde veri bilimcilerin zamanlarının önemli bir bölümünü model geliştirmeye değil, veriyi toplamaya, temizlemeye ve düzenlemeye harcadıkları sıkça dile getirilir. Gerçek projelerde en çok emek isteyen adım genellikle veridir.
Bir modelin gerçek başarısı, eğitim sırasında hiç görmediği veriler üzerindeki başarısıdır. Bu yeteneğe genelleme (generalization) denir. Genellemeyi ölçebilmek için veri seti en baştan üç parçaya ayrılır
- Eğitim kümesi (training set): Modelin parametrelerini öğrenmek için kullanılır. - Doğrulama kümesi (validation set): Model seçimi ve hiperparametre ayarı (ör. polinom derecesi, öğrenme oranı) için kullanılır. - Test kümesi (test set): Yalnızca en sonda, nihai başarımı raporlamak için bir kez kullanılır.
mede kaç görüntü bulunur? (b) Bölme katmanlı (sınıf oranları korunarak) yapılırsa test kümesinde kaç zatürre görüntüsü olur?
Test verisine ait herhangi bir bilginin eğitim sürecine karışmasına veri sızıntısı (data leakage) denir. Sızıntı, modelin başarısını olduğundan yüksek gösterir ve model gerçek hayatta beklenenden kötü çalışır. Sık görülen örnekler:
- Aynı hastaya ait farklı görüntülerin hem eğitim hem test kümesine düşmesi, - Test kümesine bakarak model veya hiperparametre seçmek.
Bir model eğitim verisindeki gürültüyü ve rastlantısal ayrıntıları da ezberlerse eğitim verisinde çok başarılı, yeni verilerde ise başarısız olur. Bu duruma aşırı öğrenme (overfitting) denir. Tersine, model verideki ilişkiyi yakalayamayacak kadar basitse hem eğitim hem test verisinde başarısız olur; buna
Sınava yalnızca geçmiş yılların sorularını cevaplarıyla ezberleyerek hazırlanan bir öğrenci, aynı sorular gelirse tam puan alır; soruların sayıları biraz değiştirildiğinde ise çözemez. Bu öğrenci aşırı öğrenmiştir. Konuyu hiç çalışmayan öğrenci ise her durumda başarısızdır (eksik öğrenme). İyi bir model, konuyu anlayan öğrenci gibi yeni sorularda da başarılı olandır.
Aşırı öğrenmeyle mücadele yöntemlerini (düzenlileştirme, çapraz doğrulama) 5. haftada, derin ağlara özgü yöntemleri (dropout, veri artırma) 7. ve 10. haftalarda ayrıntılı olarak inceleyeceğiz.
Gerçek veri setlerinde bazı hücreler boş olabilir (anket sorusunu cevaplamayan kişi, arızalanan sensör). Başlıca çözümler: (i) eksik değer içeren satırları silmek (veri azsa kayıp büyüktür), (ii) sayısal sütunlarda eksik değeri sütunun ortalaması veya medyanı ile doldurmak, (iii) kategorik sütunlarda en sık görülen değerle (mod) doldurmak. Aykırı değerlerin bulunduğu sütunlarda medyan, ortalamaya göre daha güvenilirdir.
arasında değişir. Uzaklık hesabına dayanan yöntemlerde (k-NN, k-means) ve gradyan inişle eğitilen modellerde büyük ölçekli öznitelik baskın hâle gelir. Bu nedenle öznitelikler benzer aralıklara
yonu ve (b) z-skoru standardizasyonu uygulayınız. (c) Test kümesinden gelen 50 yaşındaki bir kişinin her iki yöntemle dönüştürülmüş değerini bulunuz.
Modeller sayılarla çalıştığı için "Iğdır", "Kars", "Van" gibi kategorik değerlerin sayıya dönüştürülmesi gerekir. İlk akla gelen yöntem her kategoriye bir tam sayı vermektir (etiket kodlama (label
veriler için doğru yöntem one-hot kodlamadır (one-hot encoding): Her kategori için ayrı bir sütun açılır ve yalnızca ilgili sütun 1, diğerleri 0 olur.
Aşağıdaki tabloda bir değer eksiktir. (a) Eksik "Yaş" değerini ortalama ile doldurunuz. (b) "Şehir" sütununu one-hot kodlayınız.
Ön işlemenin son adımı, özniteliklerin birbiriyle ve etiketle nasıl ilişkili olduğunu incelemektir. İki sayısal değişken arasındaki doğrusal ilişkinin yönünü ve gücünü Pearson korelasyon katsayısı (Pe-
gösterir. Korelasyon matrisi (her öznitelik çifti için r), iki durumda yol göstericidir: Etiketle güçlü korelasyonu olan öznitelikler iyi birer aday tahmin edicidir; birbiriyle çok yüksek korelasyonlu iki öznitelik (ör. metre ve santimetre cinsinden boy) ise aynı bilgiyi taşıdığından biri atılabilir.
olarak açıklanabildiğini söyler; 3. haftada aynı veriyle kuracağımız regresyon doğrusunun R2 değeri de tam olarak 0.6 çıkacaktır.
ama doğrusal olmayan bir ilişkide r sıfıra yakın çıkabilir. Bu nedenle korelasyon her zaman saçılım grafiğiyle birlikte yorumlanmalıdır.
Sınıflandırma problemlerinde sınıfların örnek sayıları çoğu zaman çok farklıdır: Kredi kartı işlemlerinin binde biri sahtedir, tıbbi taramaların küçük bir kısmı hastalık içerir. Bu tür dengesiz veri (imbalanced data) setlerinde model, çoğunluk sınıfını tahmin ederek yüksek doğruluğa ulaşabilir ve azınlık sınıfını hiç öğrenmeyebilir. Başlıca çözümler:
- Yeniden örnekleme: Çoğunluk sınıfından örnek atmak (undersampling) veya azınlık sınıfını çoğaltmak (oversampling). SMOTE yöntemi, azınlık örneklerinin arasında ara noktalar üreterek sentetik örnekler oluşturur. Yeniden örnekleme yalnızca eğitim kümesine uygulanır; test kümesi gerçek dağılımı yansıtmalıdır. - Sınıf ağırlıkları: Kayıp fonksiyonunda azınlık sınıfının hataları daha ağır cezalandırılır. Yaygın
Yüzlerce özniteliği olan bir veri setinde öznitelikler çoğu zaman birbirine bağımlıdır ve veri aslında daha düşük boyutlu bir yapıda yaşar. Temel bileşen analizi (principal component analysis, PCA), veriyi varyansın en büyük olduğu yönlere (temel bileşenlere) izdüşürür: İlk bileşen verinin en çok yayıldığı doğrultudur, ikinci bileşen ona dik ve kalan varyansı en çok açıklayan doğrultudur; bu böyle devam eder. Bileşenler, standartlaştırılmış verinin kovaryans matrisinin özvektörleridir; her bileşenin açıkladığı varyans oranı ilgili özdeğerle orantılıdır. Uygulamada ilk k bileşen alınarak (ör. toplam varyansın %95'ini açıklayan sayıda) veri d boyuttan k boyuta indirgenir. PCA; görselleştirme (2-3 boyuta indirgeme), gürültü azaltma ve uzaklık tabanlı yöntemleri hızlandırma amacıyla kullanılır. Doğrusal olmayan yapılar için t-SNE ve UMAP gibi görselleştirme yöntemleri tercih edilir.
Veri setinde satırlar örnek, sütunlar öznitelik, tahmin edilen sütun etikettir. Denetimli öğrenmede etiketler bilinir; etiket kategorikse sınıflandırma, sürekliyse regresyon
Denetimsiz öğrenme etiketsiz veride yapı arar (kümeleme, boyut indirgeme); pekiştirmeli öğrenmede ajan ödül sinyaliyle öğrenir.
melerine ayrılır. Test bilgisinin eğitime karışması veri sızıntısıdır. Aşırı öğrenmede eğitim hatası çok düşük, test hatası yüksektir; eksik öğrenmede ikisi de
eğitim kümesinden hesaplanır. Nominal kategorik değişkenler one-hot kodlanır; etiket kodlama sahte bir sıra ilişkisi yara-
Dengesiz veride yeniden örnekleme (yalnızca eğitim kümesinde), sınıf ağırlıkları ve doğruluk dışı metrikler kullanılır. PCA, veriyi en çok varyans taşıyan yönlere izdüşürerek boyut
1. Bir bankanın müşterilerini, herhangi bir etiket kullanmadan harcama alışkanlıklarına göre gruplara ayırması hangi tür bir problemdir?
2. Aşağıdakilerden hangisi bir regresyon problemidir? A Bir tweet'in olumlu ya da olumsuz olduğunu belirlemek B Bir yaprak fotoğrafındaki hastalık türünü belirlemek
6. Aşağıdakilerden hangisi bir veri sızıntısı örneğidir? A Veriyi eğitim ve test kümelerine katmanlı olarak bölmek B Standardizasyon için ortalama ve standart sapmayı, veriyi bölmeden önce tüm veri
C Hiperparametreleri doğrulama kümesine göre seçmek D Eksik değerleri eğitim kümesinin medyanıyla doldurmak E Test kümesini yalnızca en sonda bir kez kullanmak
A Eksik öğrenme; modeli daha karmaşık hâle getirmek B Aşırı öğrenme; daha fazla veri toplamak veya modeli basitleştirmek C Eksik öğrenme; eğitimi erken durdurmak D Aşırı öğrenme; test kümesini eğitime eklemek E Model idealdir; değişiklik gerekmez
11. Eğitim, doğrulama ve test kümelerinin her birinin görevini birer cümleyle açıklayınız. Test kümesi neden model seçiminde kullanılmamalıdır?
ruluğu nedir ve bu model neden işe yaramaz? (b) Çoğunluk sınıfı, sahte işlemlerin dört katı olacak şekilde alt örneklenirse eğitim kümesi kaç örnek içerir? (c) Yeniden örnekleme neden test kümesine uygulanmaz?
Bu haftanın sonunda öğrenciler; - basit ve çoklu doğrusal regresyon modelini yazabilecek ve parametrelerini yorumlayabile-
- artık (hata) ve kayıp fonksiyonu kavramlarını açıklayabilecek, - en küçük kareler yöntemiyle regresyon katsayılarını elle hesaplayabilecek, - MSE, RMSE, MAE ve R2 metriklerini hesaplayıp yorumlayabilecek, - gradyan iniş algoritmasının adımlarını uygulayabilecek ve öğrenme oranının etkisini açık-
- en küçük kareler çözümünü normal denklemle matris formunda bulabilecek ve ölçeklemenin gradyan inişin yakınsamasına etkisini açıklayabilecektir.
Regresyon, sürekli bir hedef değişkeni tahmin etmeyi amaçlayan denetimli öğrenme görevidir. Bu ailenin en basit ve en çok kullanılan üyesi doğrusal regresyondur (linear regression). Tek öznitelikli durumda model bir doğrudur:
linde modellenebilir. Bu modelde bina yaşının katsayısının negatif olması, diğer her şey sabitken eski binaların daha ucuz olduğunu söyler. Katsayıların bu şekilde doğrudan yorumlanabilmesi, doğrusal regresyonun en büyük avantajlarından biridir.
bir parabol çizse de w1 , w2 , b parametrelerine göre doğrusal olduğu için yine doğrusal regresyonla çözülür. Buna polinom regresyon (polynomial regression) denir; 2. haftadaki Şekil 2.6 bu
Artıkları tek bir sayıya indirgeyen fonksiyona kayıp fonksiyonu (loss function) denir. Regresyonda en yaygın kayıp, ortalama kare hatadır (mean squared error, MSE):
Ayrıca kare fonksiyonunun türevi kolay alındığından optimizasyon kolaylaşır. Alternatif olarak hataların mutlak değerinin ortalaması (MAE) da kullanılabilir (Şekil 3.2).
Basit doğrusal regresyonda MSE'yi en küçük yapan w ve b değerleri kapalı formda (doğrudan formülle) bulunabilir. Bu yönteme en küçük kareler (least squares) yöntemi denir.
Formül sezgisel olarak şunu söyler: x ortalamanın üzerindeyken y de ortalamanın üzerinde olma eğilimindeyse pay pozitif çıkar ve eğim pozitif olur. b'nin formülü ise en iyi doğrunun her zaman
kendisidir. MSE, (w, b)'ye göre dışbükey bir paraboloit olduğundan bu tek durağan nokta aynı zamanda küresel en küçük noktadır.
sayısı on binleri bulduğunda ya da iki öznitelik birbirinin katı olduğunda (matris tekil olur) gradyan iniş tercih edilir. Ayrıca bu kapalı çözüm yalnızca MSE kaybına ve doğrusal modele özgüdür; sinir ağlarında böyle bir formül yoktur ve tek seçenek gradyan iniştir.
R2 formülündeki pay hata kareler toplamı (sum of squared errors, SSE), payda ise toplam kareler toplamıdır (total sum of squares, SST). SST, hiçbir öznitelik kullanmadan her örnek için y tahmin eden en basit modelin hatasıdır. Dolayısıyla R2 , modelimizin bu "tembel" modele göre ne kadar iyi oldu-
Yorum: Model, proje puanlarındaki değişkenliğin %60'ını açıklamaktadır. Tahminler gerçek puanlardan tipik olarak yaklaşık 0.7 puan sapmaktadır.
ortalamayı tahmin etmekten bile kötüyse (SSE SST) R2 negatif çıkar. Negatif R2 , modelin işe yaramadığının güçlü bir işaretidir. Ayrıca yüksek R2 tek başına modelin doğru olduğunu kanıtlamaz; artıkların grafiğine de bakılmalıdır.
En küçük kareler formülü yalnızca doğrusal regresyon gibi özel modeller için vardır. Lojistik regresyonda ve sinir ağlarında kaybı en küçük yapan parametreleri veren kapalı bir formül yoktur. Bu durumda kullanılan genel amaçlı yöntem gradyan iniştir (gradient descent). Derin öğrenmedeki milyarlarca parametreli modeller de gradyan inişin türevleriyle eğitilir; bu nedenle bu bölüm dersin geri kalanı için kritik öneme sahiptir.
Yoğun siste bir dağın tepesinde olduğunuzu ve vadiye inmek istediğinizi düşünün. Etrafı göremiyorsunuz; ancak ayağınızın altındaki zeminin hangi yöne doğru eğimli olduğunu hissedebiliyorsunuz. Mantıklı strateji, en dik iniş yönünde küçük bir adım atmak ve bunu tekrarlamaktır. Gradyan iniş tam olarak budur: Dağ kayıp fonksiyonu, konumunuz parametreler, eğim gradyan, adım uzunluğunuz ise öğrenme oranıdır.
Burada (eta) öğrenme oranıdır (learning rate). Adımlar, parametreler artık anlamlı ölçüde değişmeyene (yakınsama) ya da belirlenen iterasyon sayısına ulaşılana kadar tekrarlanır.
tif olduğu için (fonksiyon bu bölgede azalan) w artırılmıştır. En küçük noktaya yaklaştıkça gradyan küçüldüğünden adımlar da kendiliğinden küçülür.
Öğrenme oranı, eğitimin başarısını belirleyen en önemli hiperparametredir (hyperparameter). Hiperparametre, modelin veriden öğrendiği bir değer değil, eğitimden önce bizim belirlediğimiz bir ayardır. Şekil 3.3 aynı fonksiyonda üç farklı öğrenme oranının davranışını göstermektedir.
öğrenme oranıyla ilerleme çok yavaştır. (b) Uygun öğrenme oranıyla birkaç adımda en küçük noktaya ulaşılır. (c) Çok büyük öğrenme oranıyla adımlar en küçük noktanın üzerinden atlar ve
- çok küçükse: Algoritma doğru yönde ilerler ama yakınsamak için çok fazla iterasyon gerekir. - uygunsa: Kayıp hızla ve kararlı biçimde azalır. - çok büyükse: Algoritma vadinin bir yamacından diğerine sıçrar; salınım yapar ya da ıraksar
Öznitelikler farklı ölçeklerdeyse kayıp yüzeyi bir yönde dik, diğer yönde düz uzamış bir vadiye dönüşür. Gradyan dik yönde büyük olduğu için adımlar vadinin duvarları arasında zikzak çizer; öğrenme oranı dik yöne göre küçük seçilmek zorunda kalır ve düz yönde ilerleme çok yavaşlar.
1a gerekir. Öznitelik 10 kat büyütülürse a 100 kat büyür ve izin verilen en büyük öğrenme oranı 100 kat küçülür. Z-skoru standardizasyonu (2. hafta) tüm yönlerdeki eğrilikleri birbirine yaklaştırır; böylece tek bir öğrenme oranı her yönde iyi çalışır ve yakınsama hızlanır. Bu nedenle gradyan inişle eğitilen her modelde öznitelik ölçekleme zorunlu bir ön adımdır.
- Yığın (batch) gradyan iniş: Her adımda tüm eğitim verisi kullanılır. Kararlıdır ancak büyük veride her adım çok pahalıdır. - Stokastik gradyan iniş (stochastic gradient descent, SGD): Her adımda rastgele seçilen tek örnek kullanılır. Adımlar ucuz ama gürültülüdür.
Gradyan iniş fikri yeni değildir: Fransız matematikçi Augustin-Louis Cauchy 1847'de denklem sistemlerini çözmek için bu yöntemi önermiştir. Bugün ChatGPT gibi modelleri eğitirken kullanılan Adam optimizasyon algoritması da özünde Cauchy'nin fikrinin akıllıca geliştirilmiş bir
A Model tüm örnekleri hatasız tahmin etmektedir. B Model, her örnek için hedefin ortalamasını tahmin etmekten daha iyi değildir. C Modelin MSE değeri sıfırdır. D Model aşırı öğrenmiştir. E Öznitelikler ile hedef arasında mükemmel doğrusal ilişki vardır.
4. Gradyan iniş algoritmasında parametreler hangi yönde güncellenir? A Gradyan ile aynı yönde B Gradyanın tersi yönünde
10. Öğrenme oranının çok küçük ve çok büyük seçilmesinin sonuçlarını birer cümleyle açıklayınız. Öğrenme oranı neden "parametre" değil "hiperparametre" olarak adlandırılır?
Bu haftanın sonunda öğrenciler; - ikili, çok sınıflı ve çok etiketli sınıflandırma problemlerini ayırt edebilecek, - lojistik regresyonda sigmoid fonksiyonuyla olasılık hesaplayıp karar verebilecek ve ikili çapraz entropi kaybını hesaplayabilecek, - k-en yakın komşu algoritmasını uzaklık hesabıyla elle uygulayabilecek, - karar ağaçlarında entropi, Gini ve bilgi kazancı hesaplayarak bölme özniteliğini seçebile-
- Naive Bayes sınıflandırıcısıyla sonsal olasılık hesaplayabilecek, - destek vektör makinelerinin maksimum marj fikrini açıklayabilecek ve algoritmaları güçlü-zayıf yönleriyle karşılaştırabilecek, - çapraz entropi kaybının gradyanını türetebilecek, sürekli bir öznitelik için Gini ile en iyi eşiği bulabilecek ve SVM marjını hesaplayabilecektir.
Sınıflandırmada amaç, bir örneği önceden tanımlı sınıflardan birine (ya da birkaçına) atamaktır. Model, öznitelik uzayını sınıflara karşılık gelen bölgelere ayırır; bu bölgeleri ayıran yüzeye karar sınırı (decision boundary) denir. Sınıflandırma problemleri üç türe ayrılır:
- İkili sınıflandırma (binary classification): İki sınıf vardır. Spamnormal, hastasağlıklı, geçtikaldı. Sınıflar genellikle 1 (pozitif) ve 0 (negatif) olarak kodlanır. - Çok sınıflı sınıflandırma (multi-class classification): İkiden fazla sınıf vardır ve her örnek tam olarak bir sınıfa aittir. El yazısı rakam tanıma (10 sınıf), bitki hastalığı türü. - Çok etiketli sınıflandırma (multi-label classification): Bir örnek aynı anda birden fazla etikete sahip olabilir. Bir fotoğrafta hem "kedi" hem "kanepe" bulunabilir; bir film hem "komedi" hem
Bu hafta dört temel yaklaşımı inceleyeceğiz: olasılık üreten doğrusal bir model (lojistik regresyon), benzerliğe dayanan bir yöntem (k-NN), kurallar üreten bir yöntem (karar ağacı) ve olasılık ku-
lenmiştir. (a) Haftada 3 saat ve 5 saat çalışan öğrencilerin geçme olasılıklarını hesaplayınız ve 0.5 eşiğine göre sınıflandırınız. (b) Karar sınırını bulunuz.
Lojistik regresyonda MSE yerine ikili çapraz entropi (binary cross-entropy) kaybı kullanılır. Bu kayıp, doğru sınıfa verilen olasılığın negatif logaritmasıdır:
Yani güncelleme miktarı, tahmin edilen olasılık ile gerçek etiket arasındaki farkla orantılıdır. Model gradyan inişle (3. hafta) eğitilir. İkiden fazla sınıf olduğunda sigmoid yerine softmax fonksiyonu kullanılır; bunu 7. haftada göreceğiz.
Sigmoid çıktısı ile MSE birlikte kullanıldığında kayıp yüzeyi dışbükey (convex) olmaz ve birçok yerel minimum oluşabilir; ayrıca model çok yanlış olduğunda gradyanlar çok küçülür ve öğrenme yavaşlar. Çapraz entropi bu iki sorunu da ortadan kaldırır.
k-en yakın komşu (k-nearest neighbors, k-NN) algoritması, "bana arkadaşını söyle, sana kim olduğunu söyleyeyim" ilkesine dayanır. Yeni bir örneği sınıflandırmak için eğitim kümesinde ona en yakın k örnek bulunur ve bu komşular arasında en çok görülen sınıf (çoğunluk oylaması) tahmin edilir. Regresyonda ise komşuların hedef değerlerinin ortalaması alınır.
1. Yeni örnek q ile eğitim kümesindeki her örnek arasındaki uzaklığı hesapla. 2. Uzaklıkları küçükten büyüğe sırala ve en yakın k örneği seç. 3. Bu k komşunun sınıflarına bak; en sık görülen sınıfı tahmin olarak döndür.
karar sınırı girintili çıkıntılı olur ve aşırı öğrenme görülür. Büyük k ise karar sınırını aşırı yumuşatır ve küçük sınıfları "yutar" (eksik öğrenme). Uygun k değeri doğrulama kümesi veya çapraz doğrulama (5. hafta) ile seçilir. İki sınıflı problemlerde oyların eşit çıkmaması için k genellikle tek sayı
- Tahmin yavaştır: k-NN eğitim sırasında hiçbir şey öğrenmez, yalnızca veriyi saklar (tembel öğrenme (lazy learning)). Her tahminde tüm eğitim örneklerine uzaklık hesaplan-
dığından büyük veride yavaştır. - Boyut laneti: Öznitelik sayısı çok arttığında tüm noktalar birbirine benzer uzaklıklarda kalır ve "en yakın komşu" kavramı anlamını yitirir (boyutluluğun laneti (curse of dimensio-
Karar ağacı (decision tree), veriyi "evethayır" sorularıyla art arda bölerek sınıflandıran, akış şemasına benzeyen bir modeldir. Ağacın en üstündeki düğüme kök, iç düğümlere karar düğümü, en alttaki düğümlere ise yaprak denir. Her yaprak bir sınıf tahmini içerir. Karar ağaçlarının en büyük avantajı yorumlanabilir olmalarıdır: Bir tahminin neden yapıldığı kökten yaprağa giden yol izlenerek açıkla-
Ağaç kurulurken her düğümde "hangi öznitelikle bölersem ortaya çıkan gruplar en saf olur?" sorusu sorulur. Saf bir düğümde örneklerin hepsi aynı sınıftandır. Saflığı ölçmek için iki ölçü kulla-
Bir A özniteliğine göre bölmenin bilgi kazancı (information gain), ebeveyn düğümün entropisinden çocuk düğümlerin ağırlıklı ortalama entropisinin çıkarılmasıyla bulunur:
On öğrencinin devam durumu, ödev teslim durumu ve dersi geçip geçmediği aşağıda verilmiştir. Kök düğümde hangi öznitelikle bölme yapılmalıdır?
Sonuç: IG(Devam) IG(Ödev) olduğundan kök düğümde Devam özniteliği kullanılır. Aynı işlem alt düğümlerde tekrarlanarak Şekil 4.4'deki ağaç elde edilir. (Gini ile de aynı sonuç çıkar:
Yaş, gelir gibi sürekli öznitelikler "x t" biçiminde ikili sorulara dönüştürülür. Aday eşikler, sıralanmış değerlerin ardışık çiftlerinin orta noktalarıdır (yalnızca etiketin değiştiği yerlere bakmak yeterlidir); her aday için safsızlık hesaplanır ve en iyi eşik seçilir. Bu işlem her düğümde, her öznite-
lik için tekrarlanır; CART algoritması (scikit-learn'ün kullandığı) safsızlık ölçüsü olarak varsayılan biçimde Gini'yi kullanır.
Sınırlandırılmayan bir karar ağacı, her yaprakta tek bir örnek kalana kadar bölmeye devam ederek eğitim verisini ezberleyebilir. Bunu önlemek için en fazla derinlik (maxdepth), bir yapraktaki en az örnek sayısı gibi sınırlar konur ya da ağaç sonradan budanır (pruning). Tek bir ağacın kararsızlığını gidermenin güçlü bir yolu, çok sayıda ağacı bir araya getirmektir. Rastgele orman (random forest), her biri verinin rastgele bir alt örneklemiyle ve özniteliklerin rastgele bir alt kümesiyle eğitilmiş yüzlerce ağacın oylamasıyla karar verir. Birden çok modeli birleştiren bu tür yöntemlere topluluk öğrenmesi (ensemble learning) denir. Rastgele ormanlar ve gradyan artırma (XGBoost, LightGBM) yöntemleri, tablo verilerinde bugün hâlâ en başarılı yöntemler arasındadır.
İki sınıfı ayıran sonsuz sayıda doğru çizilebilir. Destek vektör makineleri (support vector machines, SVM), bu doğrular arasından iki sınıfa da en uzak olanı, yani sınıflar arasındaki boşluğu (marj (margin)) en büyük yapanı seçer (Şekil 4.6). Karar sınırına en yakın olan ve sınırın konumunu belirleyen örneklere destek vektörleri (support vectors) denir; diğer örnekler silinse bile karar sınırı değişmez. Geniş marj, yeni örneklerde daha iyi genelleme sağlar.
ile az hata arasındaki dengeyi ayarlar: Büyük C hataları ağır cezalandırır (dar marj, aşırı öğrenme riski), küçük C marjı genişletir. Veri doğrusal olarak ayrılamıyorsa SVM, çekirdek hilesi (kernel trick) ile veriyi örtük olarak daha yüksek boyutlu bir uzaya taşır ve orada doğrusal bir sınır arar. Bu sınır, orijinal uzayda eğri bir sınıra karşılık gelir. En yaygın çekirdek RBF (Gauss) çekirdeğidir. SVM'ler derin öğrenme öncesi dönemde (1995-2012) en başarılı sınıflandırıcılardandı ve küçük-orta boyutlu veri setlerinde hâlâ
Sınıflandırma ikili, çok sınıflı veya çok etiketli olabilir; model öznitelik uzayını karar sınırlarıyla bölgelere ayırır.
k-NN, en yakın k komşunun çoğunluk oyuyla karar verir; ölçekleme şarttır, küçük k aşırı öğrenmeye, büyük k eksik öğrenmeye yol açar.
4. k-NN algoritması için aşağıdaki ifadelerden hangisi yanlıştır? A Uzaklık hesabına dayandığı için özniteliklerin ölçeklenmesi önemlidir. B Eğitim aşamasında parametre öğrenmez, veriyi saklar. C k değeri büyüdükçe karar sınırı daha düzgün (yumuşak) hâle gelir. D Tahmin aşaması, eğitim kümesi büyüdükçe yavaşlar.
6. Karar ağacı kurulurken bir düğümde bölme yapılacak öznitelik hangi ölçüte göre seçilir? A En düşük bilgi kazancı B En yüksek bilgi kazancı (veya en düşük ağırlıklı safsızlık) C En fazla kategorisi olan öznitelik
8. Naive Bayes sınıflandırıcısının "naif" olarak adlandırılmasının nedeni nedir? A Yalnızca iki sınıfla çalışabilmesi B Sınıf bilindiğinde özniteliklerin birbirinden bağımsız olduğunu varsayması C Önsel olasılıkları kullanmaması D Gradyan inişle eğitilmesi E Karar sınırının her zaman doğrusal olması
10. Bir düğümde 9 "Evet" ve 5 "Hayır" örneği vardır. Bir öznitelikle bölündüğünde birinci çocukta 6 Evet-2 Hayır, ikinci çocukta 3 Evet-3 Hayır oluşmaktadır. (a) Ebeveyn düğümün entropisini, (b) bu bölmenin bilgi kazancını hesaplayınız.
şilerin ise %5'inde (yanlışlıkla) pozitif sonuç vermektedir. Testi pozitif çıkan birinin gerçekten hasta olma olasılığını Bayes teoremiyle hesaplayınız ve sonucu yorumlayınız.
yazınız, (b) her aday için ağırlıklı Gini safsızlığını hesaplayıp en iyi eşiği seçiniz, (c) tek bir eşikle tüm örneklerin neden doğru sınıflandırılamayacağını açıklayınız.
Bu haftanın sonunda öğrenciler; - karışıklık matrisini oluşturup doğruluk, kesinlik, duyarlılık, özgüllük ve F1 skorunu he-
- dengesiz veri setlerinde doğruluğun neden yanıltıcı olduğunu açıklayabilecek, - ROC eğrisi ve AUC değerini yorumlayabilecek, karar eşiğinin etkisini tartışabilecek, - k-katlı çapraz doğrulamayı uygulayabilecek, - yanlılık-varyans ödünleşimini ve L1L2 düzenlileştirmenin amacını açıklayabilecek, - k-ortalamalar (k-means) algoritmasını elle uygulayabilecek ve küme sayısını dirsek yönte-
- kesinlik-duyarlılık eğrisinden ortalama kesinliği (AP) hesaplayabilecek, hiperparametre aramasını çapraz doğrulamayla planlayabilecek ve siluet skorunu yorumlayabilecektir.
Bir sınıflandırıcının başarısını ölçmenin akla gelen ilk yolu, doğru tahminlerin oranı olan doğruluktur (accuracy). Ancak doğruluk tek başına yanıltıcı olabilir. Bin kişide 50'sinin hasta olduğu bir tarama programı düşünün. Hiçbir şey öğrenmeden herkese "sağlıklı" diyen bir model %95 doğruluk elde eder; ama tek bir hastayı bile yakalayamaz. Sınıfların sayıca çok farklı olduğu bu tür veri setlerine dengesiz veri seti (imbalanced dataset) denir ve gerçek hayattaki problemlerin çoğu (hastalık teşhisi, dolandırıcılık, arıza tespiti) bu türdendir. Bu nedenle sınıflandırıcıları farklı açılardan ölçen metriklere ihtiyaç vardır.
İkili sınıflandırmada tahminler, gerçek sınıfla karşılaştırıldığında dört gruba ayrılır. Bu dört sayıyı gösteren tabloya karışıklık matrisi (confusion matrix) denir (Şekil 5.1).
Kesinlik (precision) ve duyarlılık (recall, sensitivity) birbirini tamamlayan iki sorunun cevabıdır. Duyarlılık tıpta "hassasiyet" (sensitivity) veya "doğru pozitif oranı" (TPR) olarak da bilinir. F1 skoru (F1 score), iki metriği tek sayıda birleştirir. Harmonik ortalama kullanıldığı için metriklerden biri çok düşükse F1 de düşük çıkar; böylece "yalnızca birinde iyi" olan model ödüllendirilmez.
doğru tespit etmiş, ayrıca 30 sağlıklı kişiye yanlışlıkla "hasta" demiştir. (a) Karışıklık matrisini oluşturunuz. (b) Doğruluk, kesinlik, duyarlılık, özgüllük ve F1 değerlerini hesaplayınız. (c) Herkese "sağlıklı" diyen modelin doğruluğu ve duyarlılığı nedir?
- Duyarlılık önceliklidir (FN pahalıdır): kanser taraması, hırsızlık alarmı, salgın tespiti. Bir hastayı kaçırmak, birkaç yanlış alarmdan çok daha maliyetlidir. - Kesinlik önceliklidir (FP pahalıdır): spam filtresi (önemli bir e-postayı spam klasörüne atmak), bir müşteriyi haksız yere dolandırıcı ilan etmek, öneri sistemleri. - Sınıflar dengesizse doğruluk yerine F1, kesinlik-duyarlılık ve AUC birlikte raporlanmalı-
minleri gösterir ve köşegen doğru tahminleri içerir. Kesinlik, duyarlılık ve F1 her sınıf için "o sınıf pozitif, diğerleri negatif" kabul edilerek ayrı ayrı hesaplanır. Sonra bu değerlerin basit ortalaması alınır (makro ortalama (macro average)) ya da sınıfların örnek sayısına göre ağırlıklı ortalaması alınır (ağırlıklı ortalama). Makro ortalama, küçük sınıflara da büyük sınıflar kadar önem verdiği için dengesiz veride tercih edilir.
Lojistik regresyon gibi modeller bir olasılık üretir ve sınıf kararı bir eşikle verilir. Eşiği düşürmek modeli daha "cesur" yapar: Daha fazla örneğe "pozitif" der, duyarlılık artar ama kesinlik genellikle düşer. Eşiği yükseltmek ise tersini yapar (Şekil 5.2). Varsayılan 0.5 eşiğinin probleme göre en iyi
- AUC'nin olasılıksal anlamı: Rastgele seçilen bir pozitif örneğe, rastgele seçilen bir negatif örnekten daha yüksek skor verilme olasılığıdır.
ROC eğrisi, negatif sınıf çok büyük olduğunda iyimser görünebilir: Binlerce negatif arasında birkaç yüz yanlış pozitif, FPR'yi neredeyse hiç değiştirmez ama kesinliği çökertir. Dengesiz problemlerde bu yüzden kesinlik-duyarlılık eğrisi (precision-recall curve) tercih edilir: Eşik düşürüldükçe elde edilen (duyarlılık, kesinlik) çiftleri çizilir. Eğrinin altındaki alan ortalama kesinlik (average precision, AP) olarak özetlenir ve pratikte, her yeni doğru pozitifin yakalandığı andaki kesinliklerin ortala-
Rastgele bir modelin AP'si pozitif sınıfın oranına eşittir (%1 pozitif içeren veride 0.01); bu yüzden AP, aynı verideki AUC'den çok daha sert bir ölçüttür.
Aynı modelin AUC'si 0.8125 idi; iki metrik farklı soruları yanıtlar. Nesne tespitinde kullanılan mAP (10. hafta) de bu hesabın sınıflar üzerinden ortalamasıdır.
Veriyi bir kez eğitim ve doğrulama kümelerine ayırmak, özellikle küçük veri setlerinde şansa bağlı sonuçlar verebilir: Doğrulama kümesine kolay örnekler düşerse model olduğundan iyi görünür. k-katlı çapraz doğrulama (k-fold cross-validation) bu sorunu şöyle çözer (Şekil 5.4): Eğitim verisi k eşit parçaya (kata) bölünür. Model k kez eğitilir; her seferinde bir kat doğrulama, kalan k - 1 kat eğitim için kullanılır. Sonuçta k başarım değerinin ortalaması ve standart sapması raporlanır. Yaygın
Çapraz doğrulamanın en yaygın kullanımı hiperparametre (hyperparameter) seçimidir: k-NN'deki k, SVM'deki C, ağacın en büyük derinliği, öğrenme oranı gibi eğitimle öğrenilmeyen ayarlar. Izgara arama (grid search) her hiperparametre için bir değer listesi alır ve tüm kombinasyonları çapraz
sayısı hızla patladığından rastgele arama, belirli bir bütçe kadar rastgele kombinasyon dener ve aynı bütçede genellikle ızgaradan daha iyi sonuç verir; çünkü önemsiz hiperparametrelere eşit zaman harcamaz. Önceki denemelerden öğrenerek umut vadeden bölgelere yönelen Bayesçi optimizasyon araçları (Optuna gibi) da yaygındır.
Yüzlerce kombinasyon denendiğinde, seçilen en iyi ayar kısmen doğrulama kümesinin şansını da yansıtır; raporlanan doğrulama skoru iyimserdir. Bu yüzden nihai başarım, hiperparametre seçiminde hiç kullanılmamış test kümesinde ölçülür. Küçük veride iç içe çapraz doğrulama (dış döngü başarımı, iç döngü hiperparametreleri belirler) kullanılır.
Bir modelin test verisindeki hatası iki ana kaynaktan gelir. Yanlılık (bias), modelin gerçek ilişkiyi yakalayamayacak kadar basit olmasından kaynaklanan sistematik hatadır (eksik öğrenme). Varyans (variance) ise modelin eğitim verisindeki küçük değişikliklere aşırı duyarlı olmasından kaynak-
lanan hatadır (aşırı öğrenme). Model karmaşıklaştıkça yanlılık azalır, varyans artar. Amaç, toplam hatayı en küçük yapan dengeyi bulmaktır (Şekil 5.5).
model karmaşıklığı (polinom derecesi) Şekil 5.5Model karmaşıklığı arttıkça eğitim hatası sürekli düşer; doğrulama hatası ise önce düşer, sonra yükselir. En iyi model, doğrulama hatasının en küçük olduğu noktadadır. (Eğriler 300 farklı eğitim örneklemi üzerinden ortalanmıştır.)
Yüksek yanlılıklı bir okçunun okları hep hedefin aynı yanına, merkezden uzağa saplanır (sistematik hata). Yüksek varyanslı bir okçunun okları ise ortalamada merkeze yakın olsa da tahtanın her yanına dağılır. İyi bir model, okları merkezin etrafında sıkı bir şekilde toplayan ok-
Aşırı öğrenmeyi azaltmanın en etkili yollarından biri düzenlileştirmedir (regularization). Bu yaklaşımda kayıp fonksiyonuna büyük ağırlıkları cezalandıran bir terim eklenir:
resyona Ridge (ridge regression), L1 düzenlileştirmeli olana Lasso (lasso regression) denir. L2 tüm ağırlıkları sıfıra doğru küçültürken, L1 bazı ağırlıkları tam olarak sıfır yapar ve böylece öznitelik seçimi de gerçekleştirir. Derin öğrenmede L2 düzenlileştirme ağırlık sönümü (weight decay) olarak bilinir.
Eğitim, artık yalnızca MSE'yi değil bu toplamı küçültmeye çalışır; bu yüzden aynı uyumu daha küçük ağırlıklarla sağlayan modeller tercih edilir.
Şimdi denetimsiz öğrenmenin en bilinen algoritmasına geçelim. k-ortalamalar (k-means) algoritması, etiketsiz veriyi önceden belirlenen k sayıda kümeye ayırır. Her küme bir merkez (centroid) ile temsil edilir ve her nokta en yakın merkezin kümesine atanır.
1. k küme merkezini başlat (ör. rastgele seçilen k veri noktası). 2. Atama adımı: Her noktayı en yakın merkeze (Öklid uzaklığı) ata. 3. Güncelleme adımı: Her merkezi, kendisine atanan noktaların ortalamasına taşı. 4. Atamalar değişmeyene (merkezler sabitlenene) kadar 2. ve 3. adımları tekrarla.
Algoritma, küme içi kareler toplamını (WCSS: her noktanın kendi merkezine uzaklığının karelerinin toplamı) adım adım küçültür.
nedenle WCSS'yi en küçük yapan k aranmaz. Bunun yerine dirsek yönteminde (elbow method) WCSS, k'ye karşı çizilir ve eğrinin belirgin biçimde kırıldığı, sonrasında azalmanın yavaşladığı nokta seçilir (Şekil 5.8). Siluet skoru gibi başka ölçütler de kullanılabilir.
k-means, kümelerin yaklaşık küresel ve benzer büyüklükte olduğunu varsayar; uzun, iç içe veya farklı yoğunluktaki kümeleri iyi ayıramaz. Sonuç başlangıç merkezlerine bağlıdır; bu yüz-
den algoritma farklı başlangıçlarla birkaç kez çalıştırılır (ninit) ve en düşük WCSS'yi veren sonuç seçilir. Ölçekleme de k-NN'de olduğu gibi şarttır.
Kümeleme sonucunun kalitesini etiketsiz ölçmenin yaygın yolu siluet skorudur (silhouette score). Bir i örneği için ai kendi kümesindeki diğer örneklere ortalama uzaklık, bi ise en yakın başka kümenin örneklerine ortalama uzaklık olsun:
Tüm örneklerin ortalama siluet skoru, farklı k değerlerini karşılaştırmak için dirsek yöntemine iyi bir alternatiftir: Skoru en büyük yapan k seçilir. k-means dışında iki yöntem daha bilinmelidir. DBSCAN, yoğunluğa dayalıdır: Yarıçapı olan komşuluğunda en az minPts örnek bulunan "çekirdek" noktalar birbirine zincirlenerek kümeleri oluşturur. Küme sayısı önceden verilmez, keyfi biçimli kümeler bulunabilir ve seyrek bölgelerdeki noktalar gürültü olarak ayrılır. Hiyerarşik (birleştirici) kümeleme her örneği ayrı bir küme olarak başlatır ve en yakın iki kümeyi art arda birleştirir; sonuç, istenen seviyede kesilerek küme sayısı belirlenen ağaç biçimli bir dendrogramdır. "En yakın" tanımı bağlantı ölçütüne (tek, tam, ortalama,
Yanlılık basit modellerin, varyans karmaşık modellerin hatasıdır; L1L2 düzenlileştirme büyük ağırlıkları cezalandırarak aşırı öğrenmeyi azaltır.
Dengesiz veride kesinlik-duyarlılık eğrisi ve AP, ROCAUC'den daha bilgilendiricidir; AP, her doğru pozitif yakalandığındaki kesinliklerin ortalamasıdır. Hiperparametreler ızgararastgele aramayla ve çapraz doğrulamayla seçilir; test kümesi
A Yanlış pozitif; kesinlik B Yanlış negatif; duyarlılık C Doğru negatif; özgüllük D Doğru pozitif; doğruluk E Yanlış pozitif; doğruluk
4. Bir modelin ROC eğrisinin altında kalan alan 0.5 olarak ölçülmüştür. Bu ne anlama gelir? A Model mükemmel ayrım yapmaktadır. B Model, sınıfları rastgele tahminden daha iyi ayıramamaktadır.
6. L1 (Lasso) düzenlileştirmeyi L2 (Ridge) düzenlileştirmeden ayıran temel özellik nedir? A L1 ağırlıkları büyütür. B L1 bazı ağırlıkları tam olarak sıfır yaparak öznitelik seçimi yapar. C L1 yalnızca sınıflandırmada kullanılır. D L1 eğitim hatasını her zaman sıfırlar. E L1'de hiperparametresi yoktur.
7. k-means algoritması için aşağıdakilerden hangisi doğrudur? A Etiketli veri gerektirir. B Küme sayısı k algoritma tarafından otomatik bulunur. C Her iterasyonda merkezler, kümeye atanan noktaların ortalamasına taşınır. D Sonuç başlangıç merkezlerinden hiçbir zaman etkilenmez. E WCSS'yi en küçük yapan k değeri her zaman en iyi seçimdir.
işaretlemiş, bunların 30'u gerçekten spamdir. Karışıklık matrisini oluşturup doğruluk, kesinlik, duyarlılık ve F1 değerlerini hesaplayınız.
Hatanın baskın bileşenine göre modeli iyileştirmek için ne önerirsiniz? Daha fazla veri toplamak bu durumda ne kadar yardımcı olur?
Bu haftanın sonunda öğrenciler; - biyolojik nöron ile yapay nöron arasındaki benzerliği açıklayabilecek, - perceptronun çıktısını hesaplayabilecek ve perceptron öğrenme kuralını uygulayabilecek, - tek katmanlı perceptronun XOR problemini neden çözemediğini gösterebilecek, - başlıca aktivasyon fonksiyonlarını tanıyıp doğrusal olmayan aktivasyonun gerekliliğini
- çok katmanlı bir ağda ileri yayılım hesabını matris formunda elle yapabilecek, - bir ağın parametre sayısını hesaplayabilecek ve çıkış katmanını probleme göre seçebilecek, - ileri yayılımı bir mini-yığın için matris formunda yazabilecek ve XavierHe ağırlık başlatmasının gerekçesini açıklayabilecektir.
İnsan beyninde yaklaşık 86 milyar sinir hücresi (nöron) bulunur ve her biri binlerce başka nöronla bağlantı kurar. Bir nöron, dendritleri aracılığıyla diğer nöronlardan elektrokimyasal sinyaller alır. Bu sinyaller hücre gövdesinde birikir; toplam uyarı belirli bir eşiği aşarsa nöron "ateşlenir" ve akson boyunca bir sinyal göndererek bağlantılı olduğu nöronları uyarır. Bağlantı noktalarına sinaps denir ve öğrenme, büyük ölçüde sinapsların güçlenmesi ya da zayıflamasıyla gerçekleşir. Yapay nöron (artificial neuron), bu davranışın çok basitleştirilmiş matematiksel bir modelidir (Şekil 6.1): Girdiler xi dendritlere, ağırlıklar (weights) wi sinaps güçlerine, toplama işlemi hücre gövdesine, aktivasyon fonksiyonu (activation function) f ise ateşleme eşiğine karşılık gelir:
Burada z ağırlıklı toplam (ön aktivasyon), b bias, a ise nöronun çıktısıdır (aktivasyon). Bias terimi, nöronun ne kadar kolay "ateşleneceğini" ayarlar; geometrik olarak karar sınırının orijinden kaydı-
Yapay sinir ağları beyinden ilham almıştır, ancak beynin bir kopyası değildir. Gerçek nöronlar zamanlamaya dayalı ateşleme örüntüleri, kimyasal ileticiler ve çok daha karmaşık dinamiklerle çalışır. Modern derin öğrenme, biyolojiden çok matematik ve mühendislik ilkeleriyle
Frank Rosenblatt'ın 1958'de geliştirdiği perceptron (perceptron), aktivasyon fonksiyonu olarak basamak fonksiyonu kullanan tek bir yapay nörondur:
bu kuralın sonlu sayıda adımda tüm örnekleri doğru sınıflandıran ağırlıkları bulacağını kanıtlamıştır (perceptron yakınsama teoremi).
Perceptronun sınıfları bir doğruyla ayırabilmesi için verinin doğrusal olarak ayrılabilir (linearly separable) olması gerekir. AND ve OR böyledir; ancak girdilerden yalnızca biri 1 olduğunda 1 üreten
lılığı göstermesi, sinir ağı araştırmalarına olan ilginin uzun süre azalmasına yol açmıştır (1. hafta). Çözüm, nöronları katmanlar hâlinde art arda bağlamaktır. XOR, "OR olsun ama AND olmasın" şek-
Gizli katmanlar, veriyi problemin daha kolay çözülebildiği yeni bir temsile dönüştürür. Derin öğrenmenin gücü, bu temsillerin elle tasarlanmak yerine veriden öğrenilmesinden gelir.
Basamak fonksiyonunun türevi her yerde sıfırdır (sıfır noktasında ise tanımsızdır); bu nedenle gradyan inişle eğitim için kullanılamaz. Modern ağlarda türevlenebilir aktivasyon fonksiyonları
Aktivasyon fonksiyonu kullanmasaydık (ya da doğrusal bir fonksiyon kullansaydık) katman eklemenin hiçbir faydası olmazdı. Çünkü doğrusal dönüşümlerin bileşkesi yine doğrusal bir dönü-
Yani yüz katmanlı doğrusal bir ağ, tek katmanlı doğrusal bir modelden daha güçlü değildir ve XOR'u yine çözemez. Doğrusal olmayan aktivasyonlar, ağın eğri karar sınırları ve karmaşık ilişkiler
Nöronların katmanlar hâlinde düzenlendiği ve her katmanın bir sonrakine tam bağlı olduğu ağlara çok katmanlı perceptron (multilayer perceptron, MLP) ya da tam bağlı ileri beslemeli ağ (fully connected feedforward network) denir (Şekil 6.4). İlk katman girdi katmanı, son katman çıkış katmanı, aradakiler ise gizli katmanlardır (hidden layers). "Derin" öğrenmedeki derinlik, gizli katman sayısını ifade eder.
Girdiden başlayarak katman katman çıktıların hesaplanmasına ileri yayılım (forward propagation) denir. l. katmanın hesabı matris formunda şöyle yazılır:
Tam bağlı bir katmanda ngirdi nöronlu bir katmandan nçıktı nöronlu bir katmana geçerken her çıkış nöronunun ngirdi ağırlığı ve bir bias değeri vardır:
Bias vektörü her satıra ayrı ayrı eklenir (NumPy'daki yayınlama, broadcasting). Bu yazım, GPU'ların matris çarpımını çok hızlı yapmasının derin öğrenmede neden belirleyici olduğunu da açıklar: Döngü yerine tek bir büyük çarpım vardır.
malı normal dağılımdan çeker. Bias değerleri sıfırla başlatılır. Keras varsayılan olarak Glorot başlatmasını kullanır; PyTorch'un varsayılanı da aynı mertebede (1ngirdi ölçeğinde) bir başlatmadır, He başlatması ise açıkça seçilir. Derin ağların 2010'lara kadar eğitilememesinin nedenlerinden biri bu basit ayarın bilinmemesiydi.
Evrensel yaklaşım teoremi (universal approximation theorem), doğrusal olmayan aktivasyonlu ve tek gizli katmanlı bir ağın, yeterince nöron içermesi koşuluyla, sürekli herhangi bir fonksiyona istenilen doğrulukta yaklaşabileceğini söyler. Ancak teorem gereken nöron sayısının ne kadar büyük olacağını veya ağın bu ağırlıkları eğitimle bulup bulamayacağını garanti etmez. Pratikte derin (çok katmanlı) ağlar, aynı fonksiyonu geniş ama sığ ağlardan çok daha az parametreyle öğrenebilir; çünkü her katman bir öncekinin öğrendiği özellikleri birleştirerek hiyerarşik temsiller kurar. Şekil 6.5, gizli katman eklemenin ve nöron sayısını artırmanın karar sınırına etkisini göstermektedir.
Gizli katmanlarda genellikle ReLU kullanılırken çıkış katmanının aktivasyonu ve kayıp fonksiyonu, problemin türüne göre seçilir (Tablo 6.2). Bu tablo, derin öğrenme modellerinin tasarımında en sık başvurulan kurallardan biridir.
Doğrusal olmayan aktivasyon şarttır; aksi hâlde çok katmanlı ağ tek katmanlı doğrusal modele indirgenir. Gizli katmanlarda varsayılan seçim ReLU'dur.
Çıkış katmanı göreve göre seçilir: regresyonda doğrusal MSE, ikili sınıflandırmada sigmoid ikili çapraz entropi, çok sınıflıda softmax kategorik çapraz entropi.
9. OR kapısını gerçekleştiren bir perceptron için ağırlık ve bias değerleri öneriniz ve dört girdi için çıktıları tablo hâlinde göstererek doğrulayınız.
olan bir dağılımdan çekilirse ön aktivasyon z'nin standart sapması yaklaşık kaç olur ve bu neden sorundur? (b) He başlatmasında ağırlıkların standart sapması ne olmalıdır? (c) Tüm ağırlıklar aynı sabit değerle başlatılırsa ne olur?
Bu haftanın sonunda öğrenciler; - softmax fonksiyonuyla sınıf olasılıklarını ve kategorik çapraz entropi kaybını hesaplayabi-
- epok, yığın (batch) ve iterasyon kavramlarını ayırt edebilecek, - hesaplama grafiği üzerinde zincir kuralını uygulayarak gradyanları bulabilecek, - küçük bir ağda geri yayılım algoritmasını adım adım uygulayabilecek, - SGD, momentum ve Adam optimizasyon algoritmalarını karşılaştırabilecek, - gradyan sönümlenmesi, dropout, toplu normalizasyon ve erken durdurma kavramlarını açıklayabilecek ve PyTorch ile basit bir eğitim döngüsü yazabilecek, - Adam güncellemesini ve toplu normalizasyonu sayısal olarak uygulayabilecek; öğrenme oranı takvimlerini ve gradyan kırpmayı açıklayabilecektir.
Geçen hafta bir ağın girdiden çıktıya nasıl hesap yaptığını gördük. Bu hafta ağın ağırlıklarının veriden nasıl öğrenildiğini inceleyeceğiz. Önce çok sınıflı sınıflandırmada çıkış katmanını tamamlaya-
C sınıflı bir problemde ağın son katmanı C adet ham skor üretir; bunlara logit (logit) denir. Softmax (softmax) fonksiyonu bu skorları toplamı 1 olan pozitif olasılıklara dönüştürür:
Üstel fonksiyon her skoru pozitif yapar ve büyük skorları daha da belirgin hâle getirir; paydadaki toplam ise olasılıkların toplamını 1'e eşitler.
One-hot vektörde yalnızca doğru sınıfın elemanı 1 olduğundan toplamdan tek terim kalır: Kayıp, modelin doğru sınıfa verdiği olasılığın negatif logaritmasıdır. İki sınıflı durumda bu ifade 4. haftadaki ikili çapraz entropiye indirgenir.
tir. (a) Softmax olasılıklarını hesaplayınız. (b) Görüntü gerçekte bir kediyse kayıp nedir? (c) Görüntü gerçekte bir kuşsa kayıp nedir?
Bir sinir ağının eğitimi, aynı dört adımın binlerce kez tekrarlanmasından oluşur (Şekil 7.2): (1) bir grup örnek için ileri yayılım ile tahmin yap, (2) kaybı hesapla, (3) geri yayılım ile kaybın her parametreye göre gradyanını bul, (4) optimizasyon algoritmasıyla parametreleri güncelle.
- İterasyon (iteration): Bir yığın için yapılan tek bir ileri geri yayılım güncelleme adımı. - Epok (epoch): Eğitim kümesindeki tüm örneklerin modele bir kez gösterilmesi. Bir epoktaki iterasyon sayısı nyığın boyutu'dur.
Bir ağdaki milyonlarca parametrenin gradyanını tek tek elle türetmek imkânsızdır. Bunun yerine hesaplama, basit işlemlerden oluşan bir hesaplama grafiği (computational graph) olarak düşünülür ve türevler, analizdeki zincir kuralıyla (chain rule) grafiğin sonundan başına doğru aktarılır:
Her düğüm yalnızca kendi yerel türevini bilir; bu yerel türev, sonraki düğümden gelen gradyanla çarpılarak bir önceki düğüme iletilir.
Toplama düğümü gelen gradyanı girdilerine aynen dağıtır. Çarpma düğümü her girdiye, gelen gradyanın diğer girdiyle çarpımını iletir. ReLU düğümü ileri geçişte girdisi pozitifse grad-
Geri yayılım (backpropagation), zincir kuralının bir sinir ağına katman katman, verimli biçimde uygulanmasıdır. 1986'da Rumelhart, Hinton ve Williams tarafından yaygınlaştırılan bu algoritma, ileri yayılımda hesaplanan ara değerleri saklar ve gradyanları çıkıştan girdiye doğru tek bir geçişte hesaplar. Böylece tüm parametrelerin gradyanı, yaklaşık bir ileri yayılım maliyetinde elde edilir. Sigmoid çıkış ve ikili çapraz entropi (veya softmax ve kategorik çapraz entropi) birlikte kullanıldığında çıkış katmanındaki gradyan son derece sadedir:
PyTorch, TensorFlow ve JAX gibi derin öğrenme kütüphaneleri, yukarıda elle yaptığımız işlemi otomatik türev (automatic differentiation) ile yapar: İleri geçiş sırasında hesaplama grafiğini kaydeder ve loss.backward() çağrıldığında tüm gradyanları hesaplar. Bu sayede araştırmacılar yalnızca modelin ileri geçişini yazar.
- RMSProp: Her parametre için gradyan karelerinin hareketli ortalamasını tutar ve adımı bununla böler; böylece büyük gradyanlı parametrelerde adım küçülür, küçük gradyanlılarda büyür (parametreye özgü öğrenme oranı). - Adam: Momentum ile RMSProp'u birleştirir. Birinci moment (gradyanların ortalaması) yönü,
t. adımdaki gradyan gt olmak üzere Adam, iki hareketli ortalama tutar ve bunları başlangıçtaki sıfır değerinin etkisini gidermek için düzeltir:
İki gözlem: (i) Adam'ın adımı gradyanın büyüklüğünden neredeyse bağımsızdır ve mertebesindedir; büyük gradyanlı parametrelerde adım otomatik olarak küçülür. (ii) Üçüncü adımda gradyan işaret değiştirdiği hâlde momentum sayesinde hareket aynı yönde, ama yavaşlayarak
Öğrenme oranı takvimi: Eğitim boyunca öğrenme oranını sabit tutmak yerine kademeli olarak azaltmak (learning rate schedule) genellikle daha iyi sonuç verir. Başta büyük adımlarla hızlı ilerlenir, sona doğru küçük adımlarla en küçük noktaya yerleşilir. Büyük modellerde ilk adımlarda öğrenme oranını sıfırdan yavaşça artırmak (warm-up) ve ardından kosinüs eğrisiyle azaltmak yaygındır. İki
Geri yayılımda gradyan, her katmandan geçerken o katmanın aktivasyon türeviyle ve ağırlıklarıyla çarpılır. Sigmoid fonksiyonunun türevi en fazla 0.25'tir (Şekil 7.4). On sigmoid katmanından geçen
ve bu katmanlar öğrenemez. Bu soruna gradyan sönümlenmesi (vanishing gradient) denir. Tersine, ağırlıklar büyükse gradyanlar katmanlar boyunca katlanarak büyüyebilir (gradyan patlaması (exp-
Başlıca çözümler: gizli katmanlarda ReLU ailesi aktivasyonlar kullanmak, uygun ağırlık başlatma (ReLU için He, tanh için XavierGlorot başlatma), toplu normalizasyon, artık bağlantılar (ResNet, 10. hafta) ve gradyan patlamasına karşı gradyan kırpma (gradient clipping). Gradyan kırpma, gradyan vektörünün normu bir c eşiğini aşarsa vektörü c gg ile değiştirir;
Tüm ağırlıklar aynı değerle (ör. sıfırla) başlatılırsa bir katmandaki tüm nöronlar aynı çıktıyı üretir, aynı gradyanı alır ve aynı şekilde güncellenir. Nöronlar hiçbir zaman farklılaşamaz (simetri sorunu). Bu nedenle ağırlıklar küçük rastgele değerlerle başlatılır; bias değerleri ise sıfırla
Toplu normalizasyon (batch normalization), bir katmanın çıktılarını her mini-yığın içinde ortalaması 0, varyansı 1 olacak şekilde normalleştirir ve ardından öğrenilebilir bir ölçek () ve kaydırma () parametresi uygular. Katmanlara gelen girdilerin dağılımını kararlı tuttuğu için daha büyük öğrenme oranlarıyla daha hızlı eğitim sağlar. Transformer modellerinde bunun yerine her örneği kendi içinde normalleştiren katman normalizasyonu (layer normalization) kullanılır. Matematiksel olarak, bir mini-yığındaki N örnek için katmanın her j özniteliği ayrı ayrı normalleş-
Test sırasında tek bir örnek için yığın istatistiği olmadığından eğitim boyunca biriktirilen hareketli ortalamalar kullanılır; PyTorch'ta model.eval() çağrısı bu geçişi yapar. Katman genellikle doğrusalevrişim katmanından sonra, aktivasyondan önce yerleştirilir.
Dropout (dropout), eğitim sırasında her iterasyonda nöronların rastgele bir kısmını (ör. %20-50) geçici olarak devre dışı bırakır (Şekil 7.5). Böylece ağ belirli nöronlara aşırı güvenemez ve bilgiyi daha dağıtık biçimde temsil etmeyi öğrenir. Bu, her adımda farklı bir alt ağ eğitmeye ve test sırasında bu alt ağların ortalamasını almaya benzer. Test sırasında dropout kapatılır ve tüm nöronlar kullanılır.
Eğitim ilerledikçe eğitim kaybı sürekli azalır; doğrulama kaybı ise bir noktadan sonra yükselmeye başlar. Bu, modelin eğitim verisini ezberlemeye başladığının işaretidir. Erken durdurma (early stopping), doğrulama kaybını her epokta izler; kayıp belirli sayıda epok (patience) boyunca iyileşmezse eğitimi durdurur ve doğrulama kaybının en düşük olduğu andaki ağırlıkları geri yükler (Şekil 7.6).
Günümüzde derin öğrenme modelleri çoğunlukla PyTorch (Meta) veya TensorFlowKeras (Google) ile geliştirilir. Bu kütüphaneler tensör (çok boyutlu dizi) işlemlerini GPU'da çalıştırır, otomatik türev alır ve hazır katmanlar, kayıp fonksiyonları ve optimizasyon algoritmaları sunar. Aşağı-
verisinin bir kez kullanılmasıdır. Geri yayılım, zincir kuralını hesaplama grafiğinde çıkıştan girdiye uygular; çıkış katmanın-
Momentum salınımları sönümler; Adam, momentum ile parametreye özgü adım büyüklüğünü birleştirir ve en yaygın optimizasyon algoritmasıdır. Sigmoid türevi en fazla 0.25 olduğundan derin sigmoid ağlarında gradyan sönümlenir; ReLU, iyi başlatma, toplu normalizasyon ve artık bağlantılar bu sorunu azaltır. Dropout, erken durdurma, ağırlık sönümü ve veri artırma derin ağlarda aşırı öğrenmeyi
Adam: mt yönü, vt ölçeği belirler; adım büyüklüğü yaklaşık mertebesindedir. Öğrenme oranı takvimleri (basamaklı, kosinüs) sona doğru adımı küçültür; gradyan kırpma normu
Toplu normalizasyon her özniteliği yığın içinde standartlaştırıp , ile yeniden ölçekler; testte hareketli ortalamalar kullanılır.
6. Derin bir ağın ilk katmanlarındaki ağırlıkların eğitim boyunca neredeyse hiç değişmediği gözlenmiştir. Gizli katmanlarda sigmoid kullanıldığı biliniyorsa en olası neden ve çözüm nedir? A Aşırı öğrenme; dropout eklemek B Gradyan sönümlenmesi; ReLU aktivasyonu ve uygun başlatma kullanmak C Gradyan patlaması; öğrenme oranını artırmak D Veri sızıntısı; test kümesini değiştirmek E Eksik öğrenme; epok sayısını azaltmak
7. Dropout için aşağıdakilerden hangisi doğrudur? A Hem eğitimde hem testte aynı oranda nöron kapatılır. B Yalnızca eğitim sırasında, her adımda rastgele nöronlar kapatılır. C Ağırlıkları kalıcı olarak sıfırlar. D Eğitimi hızlandırmak için kullanılır, düzenlileştirme etkisi yoktur. E Yalnızca çıkış katmanına uygulanır.
mış gradyanı bulunuz. (c) Toplu normalizasyon neden test sırasında yığın istatistikleri yerine hareketli ortalamaları kullanır?
Bu haftanın sonunda öğrenciler; - sayısal bir görüntünün piksellerden oluşan bir matris olarak nasıl temsil edildiğini açıklayabilecek ve bellek ihtiyacını hesaplayabilecek, - RGB ve gri ton görüntüler arasında dönüşüm yapabilecek, - histogramı yorumlayabilecek; parlaklık, kontrast ve histogram eşitleme işlemlerini açıkla-
- eşikleme ile ikili görüntü elde edebilecek, - bir görüntüye çekirdek (filtre) ile evrişim uygulayabilecek ve sonucu elle hesaplayabilecek, - bulanıklaştırma, keskinleştirme, medyan filtresi ve SobelCanny kenar bulma yöntemlerini karşılaştırabilecek, - temel morfolojik işlemleri (aşındırma, genişletme, açma, kapama) açıklayabilecek, - histogram eşitleme dönüşümünü elle hesaplayabilecek ve geometrik dönüşümleri homojen koordinatlarda ifade edip çift doğrusal enterpolasyon uygulayabilecektir.
Bilgisayar için bir görüntü, sayılardan oluşan bir matristen ibarettir. Görüntünün her bir küçük karesine piksel (pixel, picture element) denir ve her piksel o noktanın parlaklığını ya da rengini sayısal
Görüntü işlemede orijin sol üst köşededir. Bir piksel I (y, x ) veya I satır, sütun şeklinde gösterilir; satır indeksi aşağı, sütun indeksi sağa doğru artar. NumPy ve OpenCV'de bir görüntünün
Renkli bir görüntü gri tona, insan gözünün yeşile en duyarlı, maviye en az duyarlı olmasını hesaba katan ağırlıklı bir ortalamayla dönüştürülür:
RGB dışında, renk tonunu parlaklıktan ayıran HSV (ton, doygunluk, değer) gibi renk uzayları da vardır; örneğin "kırmızı nesneleri bul" gibi görevlerde HSV daha kullanışlıdır.
çözünürlüklü, kanal başına 8 bitlik sıkıştırılmamış bir renkli görüntü kaç bayt yer kaplar? (c) Saniyede 30 kare kaydeden bir kamera bir dakikada ne kadar ham veri üretir?
neden JPEG, PNG, H.264 gibi yöntemlerle sıkıştırıldığını ve görüntülerle çalışan modellerin neden çok fazla hesaplama gücü gerektirdiğini açıklar.
Her pikselin yeni değerinin yalnızca kendi eski değerine bağlı olduğu işlemlere nokta işlemleri denir. Girdi piksel değeri r, çıktı s olmak üzere:
Bir görüntünün histogramı (histogram), her yoğunluk değerine sahip piksel sayısını gösteren grafiktir. Histogram görüntünün içeriği hakkında değil, ton dağılımı hakkında bilgi verir: Sola yığılmış histogram karanlık, sağa yığılmış aydınlık, dar bir aralığa sıkışmış histogram ise düşük kontrastlı bir görüntüyü gösterir. Histogram eşitleme (histogram equalization), yoğunluk değerlerini histogram tüm aralığa yayılacak şekilde yeniden dağıtarak kontrastı otomatik olarak artırır (Şekil 8.3). Tıbbi görüntülerde ve düşük ışıkta çekilmiş fotoğraflarda sıkça kullanılır. Histogram eşitleme, kümülatif histogramı dönüşüm fonksiyonu olarak kullanır. L gri seviyeli ve N piksellik bir görüntüde n j , j seviyesindeki piksel sayısı olmak üzere her k seviyesi şu yeni değere
narak komşularından ayrılmıştır. Yeni histogram tam düz olmaz (3 ve 4 seviyeleri aynı değere gider), ancak aralık çok daha iyi kullanılır. Eşitleme tüm görüntüye tek bir dönüşüm uyguladığından yerel kontrastı artırmak için görüntüyü parçalara bölüp her parçayı ayrı eşitleyen
Eşikleme, nesneyi arka plandan ayırmanın en basit yoludur ve 11. haftada göreceğimiz segmentasyonun en temel biçimidir. Eşiği elle seçmek zordur. Otsu yöntemi (Otsu's method), histogramı iki sınıfa (nesne ve arka plan) ayırırken sınıflar içi varyansı en küçük yapan eşiği otomatik olarak bulur (Şekil 8.4). Aydınlatmanın görüntü boyunca değiştiği durumlarda her bölge için ayrı eşik hesaplayan uyarlamalı eşikleme daha iyi sonuç verir.
Nokta işlemlerinin aksine çoğu görüntü işleme yöntemi, bir pikselin yeni değerini onun komşuluğuna bakarak hesaplar. Bunun temel aracı evrişim (convolution) işlemidir: Çekirdek (kernel) ya da
Bu işlem, 9. haftada göreceğimiz evrişimli sinir ağlarının (CNN) yapı taşıdır. Tek fark, CNN'lerde çekirdek değerlerinin elle seçilmek yerine veriden öğrenilmesidir.
karşılıklı elemanlar çarpılıp toplanarak çıktının sol üst elemanı (mavi) bulunur. Çekirdek bir piksel sağa kaydırılarak işlem tekrarlanır.
Matematikte ve klasik sinyal işlemede "evrişim", çekirdek 180 döndürüldükten sonra uygulanır. Denklem (8.4)'deki döndürmesiz işlemin doğru adı çapraz korelasyondur. Simetrik çekirdeklerde (ör. Gauss) iki işlem aynı sonucu verir. Derin öğrenme kütüphaneleri de döndürmesiz işlemi kullanır ve buna "evrişim" der; çekirdek zaten öğrenildiği için bu fark önemsizdir. Bu notlarda da bu kullanımı izliyoruz.
- Ortalama (kutu) filtresi: Tüm elemanları 1k2 olan çekirdek. Her pikseli komşularının ortalamasıyla değiştirir; gürültüyü azaltır ama kenarları da bulanıklaştırır.
- Gauss filtresi: Merkeze yakın piksellere daha büyük ağırlık verir; daha doğal bir bulanıklık üretir. Bulanıklığın derecesi standart sapma ile ayarlanır. Kenar bulmadan önce gürültüyü azaltmak için sıkça kullanılır. - Keskinleştirme: Merkez pikseli güçlendirip komşuları çıkaran çekirdek (ör. merkez 5, dört komşu -1) ayrıntıları belirginleştirir.
değiyle yaklaşık hesaplanır; düz bölgelerde 0, kenarların iki yanında zıt işaretli değerler verir. Keskinleştirme maskesi (unsharp masking) görüntüden bulanık kopyasını çıkararak ayrıntı-
Görüntüdeki rastgele parlak ve karanlık noktalara tuz-biber gürültüsü denir. Bu tür gürültüde medyan filtresi (median filter) çok etkilidir: Her piksel, komşuluğundaki değerlerin ortancasıyla (medyanıyla) değiştirilir. Medyan bir evrişim değildir (doğrusal değildir); ancak aykırı değerlerden etkilenmediği için hem gürültüyü temizler hem de kenarları korur (Şekil 8.7).
Ortalama filtresi gürültü noktasının etkisini komşularına yayarken (40 civarında soluk bir leke kalır), medyan filtresi aykırı değeri tamamen yok sayar ve komşulara uygun bir değer üretir.
Kenarlar, parlaklığın hızlı değiştiği yerlerdir ve nesnelerin sınırlarını belirler. Parlaklık değişimi, görüntünün gradyanı (türevi) ile ölçülür. Sobel operatörü (Sobel operator), yatay ve dikey türevleri yaklaşık olarak hesaplayan iki çekirdekten oluşur:
Gx soldan sağa parlaklık değişimini (dikey kenarları), Gy yukarıdan aşağıya değişimi (yatay kenarları) yakalar. G ise kenarın gücünü (gradyan büyüklüğünü) verir. Kenarın yönü
Canny kenar dedektörü (Canny edge detector), en yaygın kullanılan kenar bulma algoritmasıdır ve dört adımdan oluşur: (1) Gauss filtresiyle gürültüyü azalt, (2) Sobel ile gradyan büyüklüğünü ve yönünü bul, (3) kenarları tek piksel kalınlığa incelt (maksimum olmayanların bastırılması), (4) iki eşikle (histerezis) güçlü kenarları ve onlara bağlı zayıf kenarları tut. Sonuç ince, sürekli ve gürül-
- Aşındırma (erosion): Nesneleri inceltir; yapı elemanı nesnenin içine tamamen sığmıyorsa piksel silinir. Küçük beyaz gürültüleri yok eder, ince bağlantıları koparır. - Genişletme (dilation): Nesneleri kalınlaştırır; küçük delikleri doldurur, kopuk parçaları birleş-
Morfolojik işlemler özellikle eşikleme veya segmentasyon sonrasında maskeleri temizlemek için kullanılır. Örneğin bir hücre görüntüsünde eşikleme sonrası oluşan küçük gürültüler açma ile temizlenir, ardından bağlantılı bileşen analiziyle hücreler sayılır.
gulamada dönüşüm tersine uygulanır: Çıktı görüntüsündeki her piksel için kaynak konum hesaplanır ve oradaki değer alınır. Kaynak konum genellikle tam sayı değildir; değer, komşu piksellerden enterpolasyon (interpolation) ile elde edilir. En yakın komşu enterpolasyonu en yakın pikselin değerini alır (hızlı, ama basamaklı kenarlar üretir); çift doğrusal (bilinear) enterpolasyon dört komşunun uzaklıkla ağırlıklı ortalamasını alır. Sol üst köşesi (0, 0) olan birim karede köşe değerleri
olur. Geometrik dönüşümler, 10. haftada göreceğimiz veri artırma yönteminin temelidir: Eğitim görüntüleri rastgele döndürülüp ölçeklenerek modelin bu değişimlere dayanıklı olması sağlanır.
Bu hafta gördüğümüz yöntemlerde tüm çekirdekler ve eşikler insan tarafından tasarlanır. 2012 öncesinde bilgisayarlı görü sistemleri, kenar, köşe ve doku gibi bu tür elle tasarlanmış özniteliklerden (ör. SIFT, HOG) yararlanırdı. Evrişimli sinir ağları ise aynı evrişim işlemini kullanır, ancak hangi çekirdeklerin işe yarayacağını veriden öğrenir. İlginç biçimde, eğitilmiş CNN'lerin ilk katmanlarındaki çekirdekler kendiliğinden Sobel benzeri kenar dedektörlerine ve Gauss benzeri filtrelere
Histogram ton dağılımını gösterir; histogram eşitleme kontrastı artırır. Eşikleme ikili görüntü üretir; Otsu yöntemi eşiği otomatik seçer. Evrişimde çekirdek görüntü üzerinde kaydırılır, eleman eleman çarpım toplamı alınır. Dol-
A Görüntü çok karanlıktır; negatif alınmalıdır. B Görüntünün kontrastı düşüktür; histogram eşitleme uygulanmalıdır. C Görüntü gürültülüdür; medyan filtresi uygulanmalıdır. D Görüntü ikilidir; eşikleme gerekmez. E Görüntü renklidir; gri tona çevrilmelidir.
Bu haftanın sonunda öğrenciler; - tam bağlı ağların görüntülerdeki sınırlılıklarını ve CNN'lerin yerel bağlantı ile ağırlık paylaşımı fikirlerini açıklayabilecek, - çok kanallı girdiler için evrişim katmanının çıktısını elle hesaplayabilecek, - çekirdek boyutu, adım ve dolgu verildiğinde çıktı boyutunu hesaplayabilecek, - evrişim katmanının parametre sayısını hesaplayıp tam bağlı katmanla karşılaştırabilecek, - maksimum ve ortalama havuzlamayı uygulayabilecek, - tipik bir CNN mimarisini katman katman çözümleyebilecek ve PyTorch ile basit bir CNN
- bir CNN'de alıcı alanı katman katman hesaplayabilecek; GAP ile düzleştirmeyi ve bir evrişim katmanının işlem sayısını (MAC) karşılaştırabilecektir.
- Uzamsal yapı kaybolur: Görüntü düzleştirildiğinde (vektöre çevrildiğinde) hangi pikselin hangisine komşu olduğu bilgisi atılır. - Konum bağımlılığı: Sol üst köşedeki bir kediyi tanımayı öğrenen ağ, aynı kediyi sağ alt köşede gördüğünde bunu yeniden öğrenmek zorunda kalır.
Evrişimli sinir ağları (convolutional neural networks, CNN), 8. haftada gördüğümüz evrişim işlemini bir katman olarak kullanarak bu sorunları üç fikirle çözer:
Bir kenar dedektörü görüntünün her yerinde aynı işe yaradığı için ayrı ayrı ağırlıklar öğrenmeye gerek yoktur. Parametre sayısı görüntü boyutundan bağımsız hâle gelir. 3. Öteleme eşdeğerliği (translation equivariance): Girdideki bir örüntü kaydığında, öznitelik haritasındaki karşılığı da aynı miktarda kayar. Havuzlama katmanları ise küçük kaymalara karşı
Bir evrişim katmanı, her biri ayrı bir örüntüyü arayan çok sayıda filtreden (filter) oluşur. Her filtre girdi üzerinde kaydırılır ve bir öznitelik haritası (feature map) (aktivasyon haritası) üretir. Öznitelik haritası, filtrenin aradığı örüntünün görüntünün neresinde ne kadar güçlü bulunduğunu gösterir (Şekil 9.2). Katmandaki filtre sayısı, çıktının kanal (derinlik) sayısını belirler.
eleman çarpılıp tüm kanallar üzerinden toplanır ve bias eklenir; sonuç tek bir sayıdır. Dolayısıyla her filtre, girdi kaç kanallı olursa olsun tek bir öznitelik haritası üretir. Cçıktı filtre kullanılırsa çıktı
İki kanallı girdiden tek kanallı bir öznitelik haritası elde edilmiştir. Ardından genellikle ReLU uygulanır (burada tüm değerler pozitif olduğu için değişmez).
yüksekliğini yaklaşık yarıya indirir. - Dolgu (padding) P: Girdinin kenarlarına eklenen sıfır çerçevesinin kalınlığı. Dolgu olmadan her
Tam bağlı katman yaklaşık 5 400 kat daha fazla parametre gerektirir; üstelik evrişim katmanı her konumda 64 farklı öznitelik üretirken tam bağlı katman yalnızca 64 sayı üretir.
Havuzlama (pooling) katmanı öznitelik haritalarının uzamsal boyutunu küçültür. Böylece hesaplama azalır, sonraki katmanların daha geniş bir bölgeyi "görmesi" sağlanır ve küçük kaymalara karşı dayanıklılık kazanılır. Havuzlamanın öğrenilen parametresi yoktur ve her kanala ayrı ayrı uygulanır; kanal sayısını değiştirmez.
ve yükseklik yarıya iner. - Ortalama havuzlama (average pooling): Her pencerenin ortalaması alınır. - Global ortalama havuzlama (global average pooling): Her öznitelik haritasının tamamının ortala-
ve yükseklik azalır, kanal sayısı artar: Görüntü "küçük ama derin" bir temsile dönüşür. 2. Sınıflandırıcı: Son öznitelik haritaları düzleştirilir (flatten) veya global ortalama havuzlamayla vektöre çevrilir; ardından bir veya birkaç tam bağlı katman ve softmax çıkışı gelir.
Dikkat çekici bir sonuç: Evrişim katmanları toplam parametrelerin yalnızca yaklaşık %4'ünü oluşturur; parametrelerin büyük kısmı tam bağlı katmanlardadır. Modern mimariler bu yüzden düzleştirme yerine global ortalama havuzlama kullanır.
boyutlu tensörü HWC uzunluğunda bir vektöre açar; ardından gelen tam bağlı katmanın parametre sayısı bu uzunlukla çarpıldığı için çok büyür ve model girdinin boyutuna bağımlı hâle gelir. Global ortalama havuzlama (global average pooling, GAP) ise her kanalın tüm uzamsal değerlerinin ortalamasını alarak C uzunluğunda bir vektör üretir; parametresi yoktur ve girdi boyutu de-
havuzlama katmanları bu büyümeyi hızlandırır. Bu nedenle derin ağlar hiyerarşik öznitelikler öğrenir (Şekil 9.6): İlk katmanlar kenar ve renk geçişlerini, orta katmanlar doku ve basit şekilleri (göz, tekerlek, yaprak damarı), son katmanlar ise nesne parçalarını ve bütün nesneleri temsil eder. Alıcı alan katman katman hesaplanabilir. l. katmanın çekirdek boyutu k l , adımı sl olsun; rl alıcı alanı, jl ise bu katmandaki komşu iki nöronun girdide kaç piksel aralıkla yerleştiğini (etkin adım)
yalnızca 1 piksel büyütmüş, ancak etkin adımı ikiye katlayarak sonraki evrişimin katkısını iki katına çıkarmıştır; derin ağlarda alıcı alanın hızla büyümesinin nedeni budur. Alıcı alanı parametre eklemeden büyütmenin bir başka yolu genişletilmiş evrişimdir (dilated convolution):
Parametre sayısı belleği, işlem sayısı ise hızı belirler ve ikisi çok farklı olabilir. Bir evrişim katmanı, çıktının her konumunda ve her çıktı kanalında K . K . Cgirdi çarpma-toplama (multiply-
metresi vardır. Parametrelerin çoğu derin katmanlarda ve tam bağlı katmanlarda, işlemlerin çoğu ise yüksek çözünürlüklü ilk katmanlardadır. Bu yüzden ilk katmanlarda adım 2 veya havuzlama kullanılarak çözünürlük hızla düşürülür; toplam işlem sayısı (GFLOP), bir modelin telefonda çalışıp çalışamayacağını belirleyen ana ölçüttür.
PyTorch görüntüleri (yığın, kanal, yükseklik, genişlik) yani NCHW sırasıyla bekler. OpenCV ve NumPy ise görüntüyü (yükseklik, genişlik, kanal) sırasıyla tutar. Görüntüyü modele vermeden önce img.permute(2, 0, 1) ile eksenlerin yeri değiştirilmelidir; torchvision.transforms.ToTensor() bunu otomatik yapar.
CNN'ler yerel bağlantı ve ağırlık paylaşımı sayesinde görüntülerde tam bağlı ağlardan çok daha az parametreyle çalışır ve uzamsal yapıyı korur. Her filtre girdinin tüm kanallarını kapsar ve tek bir öznitelik haritası üretir; çıktının kanal sayısı filtre sayısına eşittir.
kanal sayısını değiştirir. İşlem sayısı Oh Ow K2 Cgirdi Cçıktı MAC'tır; parametre sayısıyla aynı şey değildir ve çıktı çözünürlüğüyle ölçeklenir.
5. Maksimum havuzlama katmanı için aşağıdakilerden hangisi yanlıştır? A Öğrenilebilir parametresi yoktur. B Her kanala ayrı ayrı uygulanır. C Uzamsal boyutu küçültür. D Kanal sayısını iki katına çıkarır. E Küçük kaymalara karşı dayanıklılık sağlar.
Bu haftanın sonunda öğrenciler; - AlexNet, VGG, GoogLeNet ve ResNet mimarilerinin getirdiği temel yenilikleri açıklayabi-
- artık bağlantıların gradyan akışına katkısını açıklayabilecek, - transfer öğrenmede öznitelik çıkarma ile ince ayar arasındaki farkı ve hangisinin ne zaman seçileceğini tartışabilecek, - veri artırma yöntemlerini sayabilecek ve amacını açıklayabilecek, - nesne tespitinde IoU değerini hesaplayabilecek, maksimum olmayanların bastırılması (NMS) algoritmasını uygulayabilecek ve tekiki aşamalı dedektörleri karşılaştırabilecek, - derinlemesine ayrılabilir evrişimin tasarrufunu ve bir sınıf için AP değerini hesaplayabilecek; çapa kutuları ve odak kaybının amacını açıklayabilecektir.
fazla etiketli görüntü içeren bir veri setidir. 2010-2017 yılları arasında düzenlenen ImageNet Büyük Ölçekli Görsel Tanıma Yarışması (ILSVRC), 1000 sınıflı yaklaşık 1.2 milyon eğitim görüntüsüyle bilgisayarlı görünün en önemli ölçütü oldu. Yarışmada top-5 hata, yani doğru sınıfın modelin en olası beş tahmini arasında yer almadığı görüntülerin oranı ölçülür. Şekil 10.1, 2012'de AlexNet ile başlayan derin CNN'lerin bu hatayı birkaç yıl içinde insan düzeyinin altına indirdiğini gösterir.
Krizhevsky, Sutskever ve Hinton'ın geliştirdiği AlexNet, LeNet'in fikirlerini çok daha büyük ölçekte uyguladı: 5 evrişim ve 3 tam bağlı katman, yaklaşık 60 milyon parametre. Başarısının ardındaki
yenilikler bugün standart hâle gelmiştir: sigmoidtanh yerine ReLU, aşırı öğrenmeye karşı dropout ve veri artırma, eğitimin iki GPU üzerinde paralel yürütülmesi.
araya ikinci bir ReLU girdiği için model daha ifade gücüne sahip olur. VGG-16 yaklaşık 138 milyon parametreyle oldukça ağırdır; ancak sade yapısı nedeniyle hâlâ öğretimde ve transfer öğrenmede
evrişimlerden önce kanal sayısını azaltmak için kullanılır (darboğaz, bottleneck). GoogLeNet, AlexNet'ten yaklaşık 12 kat az parametreyle (yaklaşık 7 milyon) daha yüksek doğruluk elde etmiştir.
Darboğaz tasarımı yaklaşık 8.5 kat daha az parametre kullanır. Bu tasarım ResNet-50 ve daha derin ResNet modellerinde de kullanılır.
Ağları derinleştirmek her zaman daha iyi sonuç vermez. Kaiming He ve arkadaşları, 56 katmanlı düz bir ağın 20 katmanlı bir ağdan yalnızca test kümesinde değil eğitim kümesinde de daha kötü olduğunu gözlemledi. Sorun aşırı öğrenme değil, çok derin ağların optimizasyonunun zorlaşmasıydı (bozulma problemi). Çözüm olarak önerilen ResNet, her bloğun girdisini çıktısına doğrudan ekleyen artık bağlantıları (residual skip connection) kullanır (Şekil 10.2):
Sıfırdan bir derin CNN eğitmek için milyonlarca etiketli görüntü ve günlerce GPU zamanı gerekir. Oysa çoğu gerçek projede (ör. birkaç yüz MR görüntüsü veya yaprak fotoğrafı) bu kadar veri yoktur. Transfer öğrenme (transfer learning), büyük bir veri setinde (genellikle ImageNet) önceden eğitilmiş bir modelin öğrendiği bilgiyi yeni bir göreve aktarır. Bu işe yarar, çünkü CNN'in ilk katmanlarının öğrendiği kenar, doku ve şekil dedektörleri neredeyse tüm görüntü problemlerinde ortaktır.
- Öznitelik çıkarma: Önceden eğitilmiş evrişim tabanının ağırlıkları dondurulur (güncellenmez); yalnızca yeni eklenen sınıflandırıcı katmanı eğitilir. Hızlıdır ve çok az veriyle bile çalışır. - İnce ayar (fine-tuning): Sınıflandırıcı eğitildikten sonra evrişim tabanının son katmanları (veya
Büyük modellerin tüm parametrelerini ince ayarla güncellemek çok pahalı olabilir. Son yıllarda yalnızca küçük bir parametre alt kümesini veya modele eklenen küçük modülleri eğiten parametre-verimli ince ayar (PEFT) yöntemleri geliştirilmiştir. Bunların en bilineni olan LoRA'yı 14. haftada büyük dil modelleri bağlamında göreceğiz; aynı yöntem görüntü modellerinde (ör. ViT) de kullanılmaktadır.
Veri artırma (data augmentation), eğitim görüntülerine anlamı değiştirmeyen rastgele dönüşümler uygulayarak her epokta modele biraz farklı örnekler gösterir (Şekil 10.4). Böylece model "kedinin kedi olması için sağa bakması gerekmediğini" öğrenir; aşırı öğrenme azalır ve genelleme artar. Dönüşümler yalnızca eğitim sırasında, rastgele olarak uygulanır; doğrulama ve test görüntüleri de-
çevirir; trafik işaretlerinde yatay çevirme "sola dön" işaretini "sağa dön"e çevirir; tıbbi görüntülerde renk değişimi tanısal bilgiyi bozabilir. Artırma yöntemleri alan bilgisiyle seçilmelidir.
Şimdiye kadar bir görüntüye tek bir etiket veren sınıflandırma problemini inceledik. Gerçek uygulamalarda görüntüde birden fazla nesne bulunur ve her birinin ne olduğunu ve nerede olduğunu bilmemiz gerekir. Bilgisayarlı görüde görevler, verdikleri ayrıntı düzeyine göre sıralanabilir (Şe-
merkez, genişlik ve yükseklikle ( xc , yc , w, h) biçiminde temsil edilir. Kutu koordinatlarını tahmin etmek bir regresyon, sınıfı tahmin etmek ise bir sınıflandırma problemidir; dedektörler ikisini birlikte
Tahmin edilen kutunun gerçek kutuya ne kadar iyi oturduğu kesişimbirleşim oranı (Intersection over Union, IoU) ile ölçülür:
Dedektörler genellikle aynı nesne için birbirine çok yakın birçok kutu üretir. Maksimum olmayanların bastırılması (Non-Maximum Suppression, NMS) bu tekrarları temizler:
- İki aşamalı dedektörler (R-CNN, Fast R-CNN, Faster R-CNN): Önce nesne içerebilecek aday bölgeler önerilir (region proposal), sonra her aday bölge sınıflandırılır ve kutusu düzeltilir. Genellikle daha doğrudur ama daha yavaştır. - Tek aşamalı dedektörler (YOLO, SSD, RetinaNet): Görüntüyü bir ızgaraya böler ve her hücre için kutuları ve sınıfları tek bir ileri geçişte doğrudan tahmin eder. "You Only Look Once" adından da anlaşılacağı gibi gerçek zamanlı çalışabilir (saniyede onlarca kare). Otonom araçlar, güvenlik kameraları ve tarım robotlarında yaygındır. - Transformer tabanlı dedektörler (DETR): Nesne tespitini bir küme tahmini problemi olarak ele alır; NMS ve elle tasarlanmış çapa kutularına ihtiyaç duymaz.
Dedektörler ortalama kesinlik ortalaması (mean Average Precision, mAP) ile değerlendirilir: Her sınıf için kesinlik-duyarlılık eğrisinin altındaki alan (AP) hesaplanır ve sınıflar üzerinden ortalaması
Kaçırılan nesne AP'yi doğrudan 0.25 düşürmüştür; sıralamanın sonundaki yanlış pozitifler ise hiç zarar vermemiştir. mAP, bu değerin tüm sınıflar (ve COCO'da IoU eşikleri) üzerinden
Tek aşamalı dedektörler her ızgara hücresinde önceden tanımlı birkaç çapa kutusu (anchor box) (farklı boyut ve en-boy oranlarında şablon kutular) kullanır; ağ her çapa için bir nesnelik skoru,
on binlerce çapa olup yalnızca birkaçı nesne içerdiğinden eğitim verisi aşırı dengesizdir: Kolay arka plan örnekleri kaybın tamamına hâkim olur. RetinaNet'in odak kaybı (focal loss) çözümü, çapraz entropiyi kolay örneklerde küçülten bir çarpan ekler:
Transfer öğrenme önceden eğitilmiş modelin bilgisini yeni göreve aktarır: Az veride taban dondurulur, çok veride ince ayar yapılır. Veri artırma eğitimde anlamı koruyan rastgele dönüşümlerle genellemeyi artırır.
kat azaltır; mobil mimarilerin temelidir. AP, enterpolasyonlu kesinlik-duyarlılık eğrisinin alanıdır; kaçırılan nesneler AP'yi doğrudan düşürür. Çapa kutuları şablon kutulardır; odak kaybı (1 - p) çarpanıyla kolay örnek-
B Paralel dallardan oluşan Inception modülü C Bloğun girdisini çıktısına ekleyen artık (atlama) bağlantıları D Sigmoid yerine ReLU kullanması E Derinlemesine ayrılabilir evrişim
3. Elinizde 300 adet göğüs röntgeni vardır ve ImageNet'te eğitilmiş bir modeli kullanmak istiyorsunuz. En uygun başlangıç stratejisi hangisidir? A Modeli sıfırdan, rastgele ağırlıklarla eğitmek B Evrişim tabanını dondurup yalnızca yeni sınıflandırıcıyı eğitmek, ardından gerekirse küçük öğrenme oranıyla sınırlı ince ayar yapmak C Tüm ağı büyük bir öğrenme oranıyla eğitmek D Son katmanı değiştirmeden ImageNet sınıflarıyla tahmin yapmak E Veri artırmayı test kümesine uygulamak
5. Maksimum olmayanların bastırılması (NMS) algoritmasının amacı nedir? A Görüntüdeki gürültüyü temizlemek B Aynı nesne için üretilen birbirine çok benzeyen kutulardan yalnızca en yüksek skorlu
6. Aşağıdaki veri artırma işlemlerinden hangisi el yazısı rakam (MNIST) sınıflandırmada etiketin anlamını değiştirebileceği için uygun değildir?
9. Transfer öğrenmede "öznitelik çıkarma" ile "ince ayar" arasındaki farkı açıklayınız. İnce ayarda neden küçük öğrenme oranı kullanılır?
Bu haftanın sonunda öğrenciler; - anlamsal, örnek ve panoptik segmentasyonu ayırt edebilecek, - eşikleme, bölge büyütme, havza (watershed) ve k-means gibi klasik segmentasyon yöntemlerini açıklayabilecek, - segmentasyonu piksel düzeyinde sınıflandırma olarak formüle edebilecek, - kodlayıcı-kod çözücü yapısını, transpoze evrişimi ve U-Net'teki atlama bağlantılarının ro-
- IoU (Jaccard) ve Dice katsayılarını maskeler üzerinde elle hesaplayabilecek ve piksel doğruluğunun neden yanıltıcı olabileceğini gösterebilecek, - segmentasyonda kullanılan kayıp fonksiyonlarını (piksel bazlı çapraz entropi, Dice kaybı)
- çok sınıflı karışıklık matrisinden mIoU hesaplayabilecek ve bağlantılı bileşen analizi gibi son işleme adımlarını uygulayabilecektir.
Görüntü segmentasyonu (image segmentation), bir görüntüyü anlamlı bölgelere ayırma işlemidir. Sınıflandırma "görüntüde ne var?", nesne tespiti "nesneler nerede?" sorusunu kutularla cevaplarken segmentasyon her pikselin hangi nesneye veya sınıfa ait olduğunu belirler. Sonuç, görüntüyle aynı boyutta bir maskedir (mask). Bu ayrıntı düzeyi, bir tümörün hacmini ölçmek, bir tarladaki hastalıklı alanın yüzdesini hesaplamak veya otonom bir aracın "sürülebilir yol" bölgesini bulmak gibi görevler için zorunludur.
- Anlamsal segmentasyon (semantic segmentation): Her piksele bir sınıf etiketi verilir ("para", "arka plan", "yol", "araba"). Aynı sınıftaki farklı nesneler ayırt edilmez; tüm paralar aynı renktedir. - Örnek segmentasyonu (instance segmentation): Her nesne ayrı ayrı bulunur ve maskelenir. Yan yana duran iki para iki ayrı nesne olarak çıkar; böylece nesneler sayılabilir.
- Panoptik segmentasyon (panoptic segmentation): İkisini birleştirir: Sayılabilen nesneler (araba, insan) örnek bazında, sayılamayan bölgeler (gökyüzü, yol, çimen) anlamsal olarak etiketlenir.
Derin öğrenme öncesinde segmentasyon, görüntünün düşük seviyeli özelliklerine dayanan yöntemlerle yapılırdı. Bu yöntemler bugün de veri az olduğunda, ön işleme aşamasında veya etiket
- Eşikleme (8. hafta): Nesne ile arka planın parlaklıkları belirgin biçimde farklıysa (ör. koyu zemin üzerinde parlak hücreler) en hızlı ve en basit yöntemdir. - Bölge büyütme (region growing): Kullanıcının seçtiği bir "tohum" pikselden başlanır; benzer yoğunluktaki komşu pikseller adım adım bölgeye eklenir. - Havza (watershed) algoritması (watershed): Gradyan büyüklüğü bir arazi yüzeyi gibi düşünülür; işaretçi noktalarından "su basılır" ve farklı kaynaklardan gelen suların buluştuğu sırtlar bölge sınırı olur. Birbirine değen nesneleri ayırmada etkilidir. Şekil 11.1'deki paralar bu yön-
Klasik yöntemler parlaklık ve renk gibi düşük seviyeli bilgiye dayanır. "Kedi" ile "kediyle aynı renkteki halı" arasındaki farkı bilmezler. Anlamsal düzeyde segmentasyon için görüntünün içeriğini anlayan, yani veriden öğrenen modellere ihtiyaç vardır.
pikselde softmax uygulanır ve en yüksek olasılıklı sınıf o pikselin etiketi olur. İkili segmentasyonda (ör. lezyon arka plan) tek kanallı çıktı ve piksel başına sigmoid yeterlidir.
Segmentasyonda ise çıktının girdiyle aynı çözünürlükte olması gerekir. Bu nedenle segmentasyon ağları iki bölümden oluşan bir kodlayıcı-kod çözücü (encoder-decoder) yapısı kullanır: - Kodlayıcı (encoder): Evrişim ve havuzlamayla görüntüyü küçültüp derinleştirir; "ne var?" sorusunu cevaplayan anlamsal öznitelikler çıkarır. Çoğu zaman ImageNet'te önceden eğitilmiş bir sınıflandırma ağıdır (ör. ResNet). - Kod çözücü (decoder): Öznitelik haritalarını adım adım büyüterek tekrar girdi çözünürlüğüne çıkarır; "nerede?" sorusunu cevaplar.
Kod çözücüde öznitelik haritasını büyütmek (upsampling) için iki yol vardır: (i) en yakın komşu veya çift doğrusal (bilinear) enterpolasyon gibi sabit yöntemler, (ii) öğrenilebilir transpoze evrişim (transposed convolution). Transpoze evrişimde her girdi değeri çekirdekle çarpılarak çıktıya "yayılır"; adım (S) çekirdeğin çıktıda ne kadar aralıkla yerleştirileceğini belirler. Örtüşen bölgelerde"""
print(f"{len(metin):,} karakter, {len(set(metin))} farklı karakter")
print(metin[:160])

99,513 karakter, 86 farklı karakter
Yapay zekâ, son on yılda bilgisayar biliminin en hızlı gelişen alanı oldu. Telefonunuzun kilidini açan yüz tanıma sistemi, e-posta kutunuzdaki istenmeyen posta 


## P1. Sıfırdan BPE tokenleştirici
BPE (byte pair encoding), metinde en sık yan yana gelen iki sembolü art arda birleştirerek bir alt kelime sözlüğü öğrenir (12. hafta, Çözümlü Örnek 12.1). `_` kelime sonunu gösterir.

In [3]:
import re, collections

def birlestir(parca, a, b):          # (a, b) çiftini tek sembol yap
    p, i = [], 0
    while i < len(parca):
        if parca[i:i + 2] == (a, b): p.append(a + b); i += 2
        else: p.append(parca[i]); i += 1
    return tuple(p)

def bpe_egit(s, birlesme=600):
    sayim = collections.Counter(re.findall(r"\w+", s.lower()))
    sozluk = {tuple(k) + ("_",): n for k, n in sayim.items()}
    kurallar = []
    for _ in range(birlesme):
        ciftler = collections.Counter()  # çift → kelime sıklığı
        for parca, n in sozluk.items():
            for a, b in zip(parca, parca[1:]): ciftler[a, b] += n
        (a, b), _ = ciftler.most_common(1)[0]     # en sık çift
        kurallar.append((a, b))
        sozluk = {birlestir(p, a, b): n for p, n in sozluk.items()}
    return kurallar

### Türkçe metinle eğitim ve yeni kelimeleri tokenlara ayırma

In [4]:
def tokenle(kelime, kurallar):
    parca = tuple(kelime) + ("_",)
    for a, b in kurallar:                    # kurallar öğrenilme sırasıyla
        parca = birlestir(parca, a, b)
    return [t.rstrip("_") for t in parca if t != "_"]

K_tr = bpe_egit(metin)                       # 600 birleştirme kuralı
print("ilk 8 kural:", ["".join(k) for k in K_tr[:8]])
for k in ["öğrencilerimizin", "sınıflandırma", "evlerimizden"]:
    print(f"{k:>16} →", " | ".join(tokenle(k, K_tr)))

ilk 8 kural: ['r_', 'n_', 'e_', 'la', 'i_', 'le', 'k_', 'a_']
öğrencilerimizin → öğren | ci | ler | im | iz | in
   sınıflandırma → sınıflandır | ma
    evlerimizden → ev | ler | im | iz | den


### İngilizce bir metinle ikinci bir tokenleştirici
İngilizce metin olarak Python'un kendi modül belgelerini (docstring) kullanıyoruz; internet gerekmez. (Python sürümüne göre metin ve sayılar biraz değişebilir.)

In [5]:
import importlib, inspect
ing = []
for ad in ["os", "re", "json", "random", "logging", "argparse", "collections", "itertools", "functools",
           "pathlib", "subprocess", "threading", "csv", "decimal", "string", "shutil", "pickle", "enum",
           "typing", "unittest", "email", "zipfile", "hashlib", "socket", "sqlite3", "difflib", "inspect"]:
    for _, nesne in inspect.getmembers(importlib.import_module(ad)):
        if (d := inspect.getdoc(nesne)): ing.append(d)
ingilizce = "\n".join(dict.fromkeys(ing))[:100_000]   # tekrarsız, ≈100 bin karakter
K_en = bpe_egit(ingilizce)
print("İngilizce ilk 8 kural:", ["".join(k) for k in K_en[:8]])

İngilizce ilk 8 kural: ['e_', 's_', 'th', 'in', 't_', 'd_', 'r_', 'n_']


### Aynı anlam, farklı token sayısı
Beş cümle çiftinin token sayılarını iki tokenleştiriciyle karşılaştırıyoruz.

In [6]:
tr = ["Yapay zekâ sistemleri verilerden öğrenir",
      "Öğrencilerimizin sınavlarında başarılar dileriz",
      "Modeli eğitmek için büyük bir veri kümesine ihtiyacımız var",
      "Bu fonksiyon listedeki en büyük değeri döndürür",
      "Evlerimizden çıkarken kapıları kilitlemeliyiz"]
en = ["Artificial intelligence systems learn from data",
      "We wish our students success in their exams",
      "We need a large dataset to train the model",
      "This function returns the largest value in the list",
      "We should lock the doors when leaving our homes"]

def say(cumleler, K):                  # toplam token sayısı
    return sum(len(tokenle(k, K)) for c in cumleler
               for k in re.findall(r"\w+", c.lower()))

for K, ad in [(K_tr, "Türkçe BPE"), (K_en, "İngilizce BPE")]:
    t, e = say(tr, K), say(en, K)
    print(f"{ad:>13}: Türkçe {t} | İngilizce {e} token",
          f"| TR/EN = {t / e:.2f}")

   Türkçe BPE: Türkçe 85 | İngilizce 130 token | TR/EN = 0.65
İngilizce BPE: Türkçe 156 | İngilizce 90 token | TR/EN = 1.73


## P6. Eğitim belleği ve maliyet hesaplayıcı
Varsayımlar: H100 GPU'nun bf16 yoğun tepe hızı ≈ 989 TFLOP/s, bunun %40'ı kullanılabiliyor (MFU), GPU-saati 2 $ (örnek fiyat). Eğitimde parametre başına ≈ 16 bayt (16 bit ağırlık + gradyan, 32 bit ana ağırlık + Adam'ın iki momenti).

In [7]:
H100, MFU, UCRET = 989e12, 0.4, 2.0          # FLOP/s, verim, $/GPU-saat (varsayım)

def hesap(N, D=None):
    D = D or 20 * N                           # Chinchilla: parametre başına ≈ 20 token
    C = 6 * N * D                             # eğitim hesabı (FLOP)
    saat = C / (H100 * MFU) / 3600            # GPU-saat
    return D, C, saat

print(f"{'model':>14} | {'çıkarım':>8} | {'eğitim belleği':>14} | {'token D':>7} | "
      f"{'hesap C':>7} | {'GPU-saat':>10} | {'maliyet':>14}")
for ad, N, D in [("7B", 7e9, None), ("70B", 70e9, None), ("Llama 3.1 405B", 405e9, 15.6e12)]:
    D, C, saat = hesap(N, D)
    print(f"{ad:>14} | {2*N/1e9:5.0f} GB | {16*N/1e9:11,.0f} GB | {D:.1e} | {C:.1e} | "
          f"{saat:10,.0f} | {saat*UCRET:12,.0f} $")

         model |  çıkarım | eğitim belleği | token D | hesap C |   GPU-saat |        maliyet
            7B |    14 GB |         112 GB | 1.4e+11 | 5.9e+21 |      4,129 |        8,257 $
           70B |   140 GB |       1,120 GB | 1.4e+12 | 5.9e+23 |    412,875 |      825,750 $
Llama 3.1 405B |   810 GB |       6,480 GB | 1.6e+13 | 3.8e+25 | 26,617,796 |   53,235,592 $


## P2. Küçük bir GPT'yi sıfırdan ön eğitmek
Anlaşılır kalması için tokenlarımız **karakterler**: sözlük, metindeki farklı karakterlerden oluşur.

In [8]:
import torch, torch.nn as nn, torch.nn.functional as F, time, copy, math
torch.manual_seed(0)
harfler = sorted(set(metin)); V = len(harfler)        # karakter sözlüğü
stoi = {c: i for i, c in enumerate(harfler)}
kodla = lambda s: [stoi[c] for c in s]                # metin → token kimlikleri
coz = lambda l: "".join(harfler[i] for i in l)        # kimlikler → metin
veri = torch.tensor(kodla(metin))
n = int(0.9 * len(veri))
egt, dog = veri[:n], veri[n:]                          # %90 eğitim, %10 doğrulama
print("sözlük:", V, "| eğitim:", len(egt), "| doğrulama:", len(dog))
print("“Yapay zekâ” →", kodla("Yapay zekâ"))

sözlük: 86 | eğitim: 89561 | doğrulama: 9952
“Yapay zekâ” → [46, 48, 63, 48, 72, 1, 73, 52, 58, 77]


### Transformer bloğu (13. hafta) — KV önbelleği desteğiyle

In [9]:
class Blok(nn.Module):
    def __init__(s, d, h):
        super().__init__(); s.h = h
        s.ln1, s.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.qkv, s.cikis = nn.Linear(d, 3 * d), nn.Linear(d, d)
        s.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(s, x, kv=None):                       # kv: önceki tokenların K, V'si
        B, T, D = x.shape
        q, k, v = s.qkv(s.ln1(x)).split(D, dim=2)
        bol = lambda t: t.view(B, T, s.h, D // s.h).transpose(1, 2)   # çok başlı
        q, k, v = bol(q), bol(k), bol(v)
        if kv is not None:                            # KV önbelleği: eskileri ekle
            k, v = torch.cat([kv[0], k], 2), torch.cat([kv[1], v], 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=kv is None)
        x = x + s.cikis(y.transpose(1, 2).reshape(B, T, D))           # artık bağlantı
        return x + s.ffn(s.ln2(x)), (k, v)

### Metin üretme fonksiyonu: açgözlü, sıcaklık ve top-p
Her adımda son konumun logitlerinden bir token seçilir ve dizinin sonuna eklenir (özbağlanımlı üretim).

In [10]:
@torch.no_grad()
def uret(m, istem, n=48, T=1.0, top_p=1.0, onbellek=True, tohum=0):
    g = torch.Generator().manual_seed(tohum)
    n = min(n, m.T - len(istem))                      # bağlam penceresini aşma
    idx = torch.tensor([kodla(istem)])
    logit, kv = m(idx)                                # istemi bir kez işle
    for _ in range(n):
        z = logit[0, -1]                              # son konumun logitleri
        if T == 0:
            s = z.argmax()                            # açgözlü
        else:
            p, sira = F.softmax(z / T, -1).sort(descending=True)
            p[p.cumsum(0) - p >= top_p] = 0           # top-p: kuyruğu kes
            s = sira[torch.multinomial(p / p.sum(), 1, generator=g)][0]
        idx = torch.cat([idx, s.view(1, 1)], 1)
        if onbellek: logit, kv = m(s.view(1, 1), kv, idx.shape[1] - 1)
        else:        logit, _ = m(idx)                # her adımda tüm dizi
    return coz(idx[0].tolist())

### Modeli kurmak: gömme + konum + 3 blok + çıkış katmanı

In [11]:
class MiniGPT(nn.Module):
    def __init__(s, V, d=96, h=4, L=3, T=64):
        super().__init__(); s.T = T                   # T: bağlam penceresi
        s.tok, s.poz = nn.Embedding(V, d), nn.Embedding(T, d)
        s.bloklar = nn.ModuleList(Blok(d, h) for _ in range(L))
        s.ln, s.son = nn.LayerNorm(d), nn.Linear(d, V)
    def forward(s, idx, kv=None, k0=0):               # k0: ilk tokenın konumu
        x = s.tok(idx) + s.poz(torch.arange(k0, k0 + idx.shape[1]))
        yeni = []
        for i, b in enumerate(s.bloklar):
            x, c = b(x, None if kv is None else kv[i]); yeni.append(c)
        return s.son(s.ln(x)), yeni                   # logitler, yeni önbellek

model = MiniGPT(V)
print(f"parametre sayısı: {sum(p.numel() for p in model.parameters()):,}")
print("eğitimden önce:", repr(uret(model, "Yapay zekâ ", n=40)))

parametre sayısı: 358,454
eğitimden önce: 'Yapay zekâ TaEgVM?  Bc OvöHjKj D0xvHb3şXLwşzLTNŞ?2b'


### Yığın üretme ve doğrulama kaybı
Hedef, girdinin bir sağa kaydırılmış hâlidir: her konum bir sonraki karakteri tahmin eder.

In [12]:
def yigin(kaynak, B=32, T=64, g=None):                # rastgele B parça × T karakter
    i = torch.randint(len(kaynak) - T - 1, (B,), generator=g)
    x = torch.stack([kaynak[j:j + T] for j in i])
    y = torch.stack([kaynak[j + 1:j + T + 1] for j in i])   # bir sağa kaydırılmış
    return x, y

@torch.no_grad()
def kayip_olc(m, kaynak, adet=20):                    # sabit örneklerle ortalama kayıp
    g = torch.Generator().manual_seed(123)
    return sum(F.cross_entropy(m(x)[0].flatten(0, 1), y.flatten()).item()
               for x, y in (yigin(kaynak, g=g) for _ in range(adet))) / adet

### Ön eğitim döngüsü (CPU'da birkaç dakika)

In [13]:
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
t0 = time.time()
for adim in range(1, 601):
    x, y = yigin(egt)                        # 32 × 64 karakter
    logit, _ = model(x)                      # ileri geçiş
    kayip = F.cross_entropy(logit.flatten(0, 1), y.flatten())
    opt.zero_grad(); kayip.backward(); opt.step()  # geri yayılım
    if adim % 100 == 0:
        L_dog = kayip_olc(model, dog)        # doğrulama kaybı
        print(f"adım {adim}: eğitim {kayip.item():.3f}",
              f"| doğrulama {L_dog:.3f}")
print(f"toplam süre: {time.time() - t0:.0f} s")

adım 100: eğitim 2.568 | doğrulama 2.560


adım 200: eğitim 2.248 | doğrulama 2.294


adım 300: eğitim 2.031 | doğrulama 2.087


adım 400: eğitim 1.775 | doğrulama 1.921


adım 500: eğitim 1.694 | doğrulama 1.817


adım 600: eğitim 1.565 | doğrulama 1.754
toplam süre: 37 s


### Eğitimden sonra: model ne öğrendi?

In [14]:
for istem in ["Yapay zekâ ", "Derin öğrenme ", "Bu hafta "]:
    print(repr(uret(model, istem, T=0.8, tohum=1)))

'Yapay zekâ bir algoritmaları olanır. Gesiz yakan katman her'
'Derin öğrenme için yapınız, bir sınıfı bir handa kaç kativaya '
'Bu hafta bir örneklemleri yenile karı göre daha üzeri kay'


## Çözümlü Örnek 14.3'ü doğrulamak: sıcaklık ve top-p

In [15]:
import numpy as np
z = np.array([3.0, 2.5, 2.0, 1.0, -1.0])            # güzel, sıcak, soğuk, yağmurlu, mavi
for T in [1.0, 0.5]:
    p = np.exp(z / T); p /= p.sum()
    print(f"T = {T}:", p.round(2))
p = np.exp(z); p /= p.sum()
tut = np.cumsum(p) - p < 0.9                         # top-p = 0.9
q = np.where(tut, p, 0); q /= q.sum()
print("top-p 0.9:", q.round(2))

T = 1.0: [0.47 0.29 0.17 0.06 0.01]
T = 0.5: [0.66 0.24 0.09 0.01 0.  ]
top-p 0.9: [0.51 0.31 0.19 0.   0.  ]


## P3. Kod çözme laboratuvarı
Aynı model, aynı istem; yalnızca seçim stratejisi değişiyor.

In [16]:
istem = "Sinir ağları "
for ad, T, p in [("açgözlü", 0, 1), ("T = 0.5", 0.5, 1), ("T = 1.0", 1.0, 1),
                 ("T = 1.5", 1.5, 1), ("T = 1, top-p 0.9", 1.0, 0.9)]:
    print(f"{ad:>16}: {uret(model, istem, T=T, top_p=p)!r}")
print("açgözlü iki kez aynı mı?", uret(model, istem, T=0) == uret(model, istem, T=0))
print("T = 1, iki farklı tohum:", [uret(model, istem, T=1, tohum=t)[13:35] for t in (1, 2)])

         açgözlü: 'Sinir ağları bir arama bir alını bir ağırlıklarını bir katman'


         T = 0.5: 'Sinir ağları ise köznin ağırlarını sınıfını edili hem kayıp v'


         T = 1.0: "Sinir ağları ise söyle geçişimin alfınızca heLN fearcell'in h"
         T = 1.5: 'Sinir ağları - ikidi geçerxNe yarı7sınız; 7.,hızl8dının ve te'


T = 1, top-p 0.9: 'Sinir ağları ise seçile her bundan küçür. Öznitelik her fil b'


açgözlü iki kez aynı mı? True


T = 1, iki farklı tohum: ['biryan parametre hem y', 'verisi bir filtresi bi']


## P4. KV önbelleği ile ve önbelleksiz üretim hızı

In [17]:
for onb in [False, True]:
    t0 = time.perf_counter()
    for _ in range(20):
        cikti = uret(model, "Y", n=63, T=0, onbellek=onb)
    ms = (time.perf_counter() - t0) / 20 * 1000
    print(f"önbellek = {onb!s:5}: {ms:6.1f} ms / 63 token")
a = uret(model, "Y", n=63, T=0, onbellek=False)
b = uret(model, "Y", n=63, T=0, onbellek=True)
print("iki yol aynı metni üretti mi?", a == b)
# önbelleksiz: her adımda tüm dizi (1, 2, 3, …, 64 token)
print("işlenen token: önbelleksiz", 1 + sum(range(2, 65)),
      "| önbellekli", 1 + 63)

önbellek = False:   76.1 ms / 63 token


önbellek = True :   39.9 ms / 63 token
iki yol aynı metni üretti mi? True
işlenen token: önbelleksiz 2080 | önbellekli 64


### Büyük modellerde KV önbelleği ve hız sınırı (Çözümlü Örnek 14.4, Soru 11)

In [18]:
def kv_bayt(L, d, bayt=2):                           # token başına K ve V
    return 2 * L * d * bayt

for ad, L, d, ctx in [("Örnek 14.4, 4096 token", 32, 4096, 4096),
                      ("Soru 11a, 8192 token", 24, 2048, 8192)]:
    b = kv_bayt(L, d)
    print(f"{ad}: {b:,} B/token → {b * ctx / 2**30:.2f} GB")
N = 7e9
for bit in [16, 8, 4]:                               # bant genişliği 1 TB/s
    gb = N * bit / 8 / 1e9
    print(f"{bit:>2} bit: {gb:4.1f} GB → ≤ {1000 / gb:3.0f} token/s")

Örnek 14.4, 4096 token: 524,288 B/token → 2.00 GB
Soru 11a, 8192 token: 196,608 B/token → 1.50 GB
16 bit: 14.0 GB → ≤  71 token/s
 8 bit:  7.0 GB → ≤ 143 token/s
 4 bit:  3.5 GB → ≤ 286 token/s


## P5. Nicemleme: ağırlıkları 8 ve 4 bite sıkıştırmak
Her satır kendi ölçeğiyle simetrik nicemlenir; sonra ağırlıklar geri çözülerek modelin kaybı ölçülür.

In [19]:
def nicemle(W, bit):                     # satır başına simetrik
    qmax = 2 ** (bit - 1) - 1                # int8 → 127, int4 → 7
    olcek = W.abs().amax(1, keepdim=True) / qmax
    q = (W / olcek).round().clamp(-qmax, qmax)   # tamsayılar
    return q * olcek                         # geri çözülmüş ağırlık

lin = [k for k in model.modules() if isinstance(k, nn.Linear)]
dogrusal = sum(k.weight.numel() for k in lin)
diger = sum(p.numel() for p in model.parameters()) - dogrusal
for bit in [32, 8, 4]:
    m = copy.deepcopy(model)
    for k in m.modules():
        if isinstance(k, nn.Linear) and bit < 32:
            k.weight.data = nicemle(k.weight.data, bit)
    mb = (dogrusal * bit / 8 + diger * 4) / 2**20
    L = kayip_olc(m, dog)
    print(f"{bit:>2} bit | {mb:5.2f} MB | doğrulama kaybı {L:.3f}")

32 bit |  1.37 MB | doğrulama kaybı 1.754


 8 bit |  0.39 MB | doğrulama kaybı 1.754


 4 bit |  0.23 MB | doğrulama kaybı 1.775


## P2b. Talimat ince ayarı (SFT): temel modelden küçük bir “asistana”
Ön eğitilmiş modele “Soru: … / Cevap: …” biçiminde 20 örnek gösteriyoruz. Son iki soru eğitimde **yok**; modelin yalnızca biçimi mi yoksa bilgiyi mi öğrendiğini onlarla test edeceğiz.

In [20]:
sorular = [("Yapay zekâ nedir?", "Zekâ isteyen işleri yapan sistem."),
           ("Makine öğrenmesi nedir?", "Veriden öğrenen yöntemler."),
           ("Derin öğrenme nedir?", "Çok katmanlı ağlarla öğrenme."),
           ("Token nedir?", "Metnin küçük bir parçası."),
           ("Gömme nedir?", "Tokenı temsil eden vektör."),
           ("Dikkat nedir?", "Tokenların birbirine bakması."),
           ("Kayıp nedir?", "Tahminin ne kadar yanlış olduğu."),
           ("Gradyan inişi nedir?", "Kaybı azaltan küçük adımlar."),
           ("Aşırı öğrenme nedir?", "Ezberleyip testte bozulmak."),
           ("Evrişim nedir?", "Filtreyi görüntüde kaydırmak."),
           ("Havuzlama nedir?", "Haritayı küçültüp özetlemek."),
           ("Sıcaklık nedir?", "Dağılımı sivrilten ayar."),
           ("Bağlam penceresi nedir?", "Modelin gördüğü tokenlar."),
           ("RAG nedir?", "Belge getirip isteme eklemek."),
           ("LoRA nedir?", "Küçük matrislerle ince ayar."),
           ("Halüsinasyon nedir?", "Akıcı ama yanlış bilgi."),
           ("ReLU nedir?", "Negatifleri sıfırlayan fonksiyon."),
           ("Transformer nedir?", "Dikkate dayalı mimari."),
           ("RLHF nedir?", "İnsan tercihleriyle hizalama."),
           ("Softmax nedir?", "Skorları olasılığa çevirir."),   # test
           ("Epok nedir?", "Verinin bir kez tamamen görülmesi.")]        # test
ornekler = [f"Soru: {s}\nCevap: {c}\n" for s, c in sorular[:-2]]
print(len(ornekler), "eğitim örneği; en uzun:", max(map(len, ornekler)), "karakter")
print(repr(ornekler[3]))

19 eğitim örneği; en uzun: 65 karakter
'Soru: Token nedir?\nCevap: Metnin küçük bir parçası.\n'


### SFT eğitimi: kayıp yalnızca cevap kısmında hesaplanır

In [21]:
def sft_yigin(B=16, T=64):
    X = torch.zeros(B, T, dtype=torch.long)
    Y = torch.full((B, T), -100)             # -100: kayıp yok
    for r, i in enumerate(torch.randint(len(ornekler), (B,))):
        s = kodla(ornekler[i])
        c = ornekler[i].index("Cevap:") + 6  # cevabın başladığı yer
        X[r, :len(s) - 1] = torch.tensor(s[:-1])
        Y[r, c - 1:len(s) - 1] = torch.tensor(s[c:])
    return X, Y

asistan = copy.deepcopy(model)               # temel modelin kopyası
opt = torch.optim.AdamW(asistan.parameters(), lr=1e-3)
for adim in range(1, 201):
    X, Y = sft_yigin()
    logit = asistan(X)[0].flatten(0, 1)
    kayip = F.cross_entropy(logit, Y.flatten(), ignore_index=-100)
    opt.zero_grad(); kayip.backward(); opt.step()
    if adim % 50 == 0:
        print(f"SFT adım {adim}: kayıp {kayip.item():.3f}")

SFT adım 50: kayıp 0.068


SFT adım 100: kayıp 0.007


SFT adım 150: kayıp 0.003


SFT adım 200: kayıp 0.002


### Temel model ve “asistan” aynı soruya ne diyor?

In [22]:
for soru in ["Derin öğrenme nedir?", "Token nedir?",
             "Softmax nedir?", "Epok nedir?"]:
    istem = f"Soru: {soru}\nCevap:"
    yeni = soru in ("Softmax nedir?", "Epok nedir?")
    print(soru, "(eğitimde yok)" if yeni else "")
    for ad, m in [("  temel  ", model), ("  asistan", asistan)]:
        cevap = uret(m, istem, n=40, T=0)[len(istem):]
        ilk_satir = cevap.split("\n")[0]
        print(f"{ad}: {ilk_satir!r}")

Derin öğrenme nedir? 
  temel  : ' Model bir katmanın bir sınır v'
  asistan: ' Çok katmanlı ağlarla öğrenme.'
Token nedir? 
  temel  : ' Model karar ve katmanın bir ağırlıklı '
  asistan: ' Metnin küçük bir parçası.'
Softmax nedir? (eğitimde yok)
  temel  : ' Model kenar ve karar ve karşılık ve '
  asistan: ' Filtreyi görüntüde kaydırmak.'
Epok nedir? (eğitimde yok)
  temel  : ' Model katmanın bir katmanın bir sınır v'


  asistan: ' Küçük matrislerle ince ayar.'


## Akıl yürütme modelleri: test zamanında çok örnekleme ve GRPO avantajı
Varsayım: her örnek birbirinden bağımsız ve %60 olasılıkla doğru (yanlış cevaplar aynı yanlışta birleşiyor: en kötü durum).

In [23]:
from math import comb
p = 0.6                      # tek örneğin doğru olma olasılığı
for k in [1, 3, 5, 9, 17, 33]:           # k örnek + çoğunluk oyu
    dogru = sum(comb(k, j) * p**j * (1 - p)**(k - j)
                for j in range(k // 2 + 1, k + 1))
    print(f"k = {k:>2} örnek → doğruluk {dogru:.3f}")

odul = np.array([1, 0, 1, 0, 1, 1], float)   # doğrulayıcı ödülü
A = (odul - odul.mean()) / odul.std()       # grup içi avantaj
print("ortalama ödül:", round(odul.mean(), 3))
print("avantajlar:", A.round(2))

k =  1 örnek → doğruluk 0.600
k =  3 örnek → doğruluk 0.648
k =  5 örnek → doğruluk 0.683
k =  9 örnek → doğruluk 0.733
k = 17 örnek → doğruluk 0.801
k = 33 örnek → doğruluk 0.879
ortalama ödül: 0.667
avantajlar: [ 0.71 -1.41  0.71 -1.41  0.71  0.71]


## Değerlendirme: eğitim ve doğrulama metninde şaşkınlık (perplexity)
Şaşkınlık = e^kayıp: model her adımda ortalama kaç seçenek arasında kararsız kalıyor?

In [24]:
for ad, kaynak in [("eğitim metni", egt), ("doğrulama metni", dog)]:
    L = kayip_olc(model, kaynak)
    print(f"{ad:>15}: kayıp {L:.3f} | şaşkınlık e^L = {math.exp(L):.2f}")

   eğitim metni: kayıp 1.547 | şaşkınlık e^L = 4.70


doğrulama metni: kayıp 1.754 | şaşkınlık e^L = 5.78


### Alıştırma (Değerlendirme Soruları 8, 9, 11 ve 12a)
Önce kâğıt üzerinde çözün, sonra hücreyi çalıştırıp karşılaştırın.

In [ ]:
z = np.array([2.0, 1.0, 0.0])                         # Soru 8
for T in [1, 2]:
    p = np.exp(z / T); print(f"Soru 8, T = {T}:", (p / p.sum()).round(3))
N = 13e9                                              # Soru 9
print(f"Soru 9: D ≈ {20*N:.1e} token, C ≈ {6*N*20*N:.1e} FLOP")
kv = 2 * 24 * 2048 * 2                                # Soru 11a
print(f"Soru 11a: {kv:,} B/token → 8192 token: {kv*8192/2**30:.2f} GB")
print(f"Soru 11b: {2 * 1.5e9 * 200:.0e} FLOP")
P = 1 / (1 + np.exp(-(1.2 - 1.7)))                    # Soru 12a
print(f"Soru 12a: P = {P:.3f}, kayıp = {-np.log(P):.2f}")